For this file, it will be showcasing the usage of models and the database for the purpose of the oral exam

first cell will connect and explore the database

In [1]:
# 0001_one_done.ipynb — Step 1: Connect & Explore Database

import sqlite3
import pandas as pd

# Set your local path
DB_PATH = "/Users/sebastianvinther/Desktop/Sportsmonks/db_sportmonks.db"

# Connect to the database
def get_connection():
    return sqlite3.connect(DB_PATH)

# List all tables
def list_tables():
    with get_connection() as conn:
        query = "SELECT name FROM sqlite_master WHERE type='table';"
        return pd.read_sql_query(query, conn)

# Show schema and sample rows from each table (optional)
def inspect_table(table_name, limit=5):
    with get_connection() as conn:
        schema = pd.read_sql_query(f"PRAGMA table_info({table_name});", conn)
        sample = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT {limit};", conn)
    return schema, sample

# View all table names
tables_df = list_tables()
tables_df


,name
0,continents
1,countries
2,regions
3,cities
4,leagues
5,seasons
6,schedules
7,stages
8,rounds
9,teams


for showcasing we will now:
Create a clean, minimal pipeline that:

Connects to the DB and fetches a fixture with its stats and players.

Builds a feature matrix from game + player statistics.

Trains a simple model (e.g., XGBoost or random forest).

Evaluates performance (e.g., accuracy).

Optionally: compares with bookmaker odds (ROI calc).


# Sportsmonks Database Analysis

This notebook analyzes the Sportsmonks SQLite database to understand its structure, content, and relationships. We'll inspect tables, examine data patterns, and prepare for further analysis.

## Setup and Initial Connection
First, we'll import the necessary libraries and establish a connection to the database.

In [4]:
# CELL 2 - CODE
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Database path
db_path = '/Users/sebastianvinther/Desktop/Sportsmonks/db_sportmonks.db'

# Test connection
try:
    conn = sqlite3.connect(db_path)
    print("✅ Successfully connected to the database!")
    print(f"Database path: {db_path}")
    conn.close()
except Exception as e:
    print(f"❌ Error connecting to database: {e}")

✅ Successfully connected to the database!
Database path: /Users/sebastianvinther/Desktop/Sportsmonks/db_sportmonks.db


## Database Overview
Let's get a high-level overview of what's in our database - how many tables, views, and other objects exist.
"""

In [6]:
def get_database_overview(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    print("📊 DATABASE OVERVIEW")
    print("=" * 50)
    
    # Get file size
    import os
    file_size = os.path.getsize(db_path)
    print(f"Database size: {file_size:,} bytes ({file_size/1024/1024:.2f} MB)")
    
    # Get all objects
    cursor.execute("""
        SELECT type, COUNT(*) as count 
        FROM sqlite_master 
        WHERE name NOT LIKE 'sqlite_%' 
        GROUP BY type
    """)
    objects = cursor.fetchall()
    
    print("\nDatabase objects:")
    for obj_type, count in objects:
        print(f"  {obj_type.capitalize()}s: {count}")
    
    conn.close()

get_database_overview(db_path)

📊 DATABASE OVERVIEW
Database size: 39,924,994,048 bytes (38075.44 MB)

Database objects:
  Indexs: 7
  Tables: 48
  Views: 1


## Table Discovery
Now let's see all the tables in our database and get a sense of what sports data we're working with.

In [7]:
def list_all_tables(db_path):
    conn = sqlite3.connect(db_path)
    
    # Get all tables
    tables_df = pd.read_sql_query("""
        SELECT name as table_name, 
               sql as create_statement
        FROM sqlite_master 
        WHERE type='table' 
        AND name NOT LIKE 'sqlite_%'
        ORDER BY name
    """, conn)
    
    print("📋 ALL TABLES IN DATABASE")
    print("=" * 50)
    
    for i, row in tables_df.iterrows():
        print(f"{i+1:2d}. {row['table_name']}")
    
    print(f"\nTotal tables: {len(tables_df)}")
    
    conn.close()
    return tables_df['table_name'].tolist()

tables = list_all_tables(db_path)

📋 ALL TABLES IN DATABASE
 1. bookmakers
 2. cards
 3. checkpoint
 4. cities
 5. coaches
 6. commentaries
 7. continents
 8. countries
 9. evaluation
10. events
11. expected_xg
12. fixture_odds
13. fixture_team_names
14. fixtures
15. groups
16. leagues
17. lineups
18. markets
19. news
20. odds
21. odds_cache
22. odds_checkpoint
23. penalties
24. player_statistics
25. players
26. predictions
27. processed_fixtures
28. processed_fixtures_odds
29. referees
30. regions
31. rivals
32. rounds
33. schedules
34. seasons
35. squads
36. stages
37. standings
38. statistics
39. statistics_checkpoint
40. team_name_mapping
41. team_squads
42. teams
43. top_scorers
44. topscorers
45. transfers
46. trends
47. tv_stations
48. venues

Total tables: 48


## Table Structure Analysis
For each table, let's examine its structure: columns, data types, constraints, and row counts.
"""


In [8]:
def analyze_table_structures(db_path, tables):
    conn = sqlite3.connect(db_path)
    
    print("🔍 TABLE STRUCTURE ANALYSIS")
    print("=" * 60)
    
    table_info = {}
    
    for table in tables:
        print(f"\n📊 TABLE: {table}")
        print("-" * 40)
        
        # Get column information
        columns_df = pd.read_sql_query(f"PRAGMA table_info({table})", conn)
        
        # Get row count
        row_count = pd.read_sql_query(f"SELECT COUNT(*) as count FROM {table}", conn)['count'][0]
        
        print(f"Rows: {row_count:,}")
        print(f"Columns: {len(columns_df)}")
        
        print("\nColumn Details:")
        for _, col in columns_df.iterrows():
            pk_marker = " [PK]" if col['pk'] else ""
            null_marker = " [NOT NULL]" if col['notnull'] else ""
            default_info = f" (default: {col['dflt_value']})" if col['dflt_value'] else ""
            
            print(f"  • {col['name']}: {col['type']}{pk_marker}{null_marker}{default_info}")
        
        table_info[table] = {
            'rows': row_count,
            'columns': len(columns_df),
            'column_details': columns_df
        }
    
    conn.close()
    return table_info

table_info = analyze_table_structures(db_path, tables)


🔍 TABLE STRUCTURE ANALYSIS

📊 TABLE: bookmakers
----------------------------------------
Rows: 0
Columns: 4

Column Details:
  • id: INTEGER [PK]
  • name: TEXT
  • updated_at: TEXT
  • raw: TEXT

📊 TABLE: cards
----------------------------------------
Rows: 459,278
Columns: 10

Column Details:
  • id: INT
  • fixture_id: INT
  • player_id: INT
  • player_name: TEXT
  • team_id: INT
  • team_name: TEXT
  • minute: INT
  • card_type: TEXT
  • reason: TEXT
  • updated_at: TEXT

📊 TABLE: checkpoint
----------------------------------------
Rows: 1
Columns: 5

Column Details:
  • id: INTEGER [PK]
  • processed_count: INTEGER
  • penalties_count: INTEGER
  • cards_count: INTEGER
  • timestamp: TEXT

📊 TABLE: cities
----------------------------------------
Rows: 102,231
Columns: 5

Column Details:
  • id: INTEGER [PK]
  • region_id: INTEGER
  • name: TEXT
  • updated_at: TEXT
  • raw: TEXT

📊 TABLE: coaches
----------------------------------------
Rows: 3,864
Columns: 7

Column Details:
  • i

"""
## Sample Data Inspection
Let's look at sample data from each table to understand what kind of sports information we're working with.
"""

In [9]:
def show_sample_data(db_path, tables, sample_size=3):
    conn = sqlite3.connect(db_path)
    
    print("📋 SAMPLE DATA FROM EACH TABLE")
    print("=" * 60)
    
    for table in tables:
        print(f"\n🔸 {table.upper()}")
        print("-" * 30)
        
        try:
            # Get sample data
            sample_df = pd.read_sql_query(f"SELECT * FROM {table} LIMIT {sample_size}", conn)
            
            if len(sample_df) > 0:
                print(sample_df.to_string(index=False, max_cols=10))
            else:
                print("  (No data in this table)")
                
        except Exception as e:
            print(f"  Error reading table: {e}")
    
    conn.close()

show_sample_data(db_path, tables)


📋 SAMPLE DATA FROM EACH TABLE

🔸 BOOKMAKERS
------------------------------
  (No data in this table)

🔸 CARDS
------------------------------
  id  fixture_id  player_id    player_name  team_id          team_name  minute card_type reason                       updated_at
None    19149018    8093398    Hrvoje Ilić     6403 Kryvbas Kryvyi Rih      44    Yellow   Foul 2025-05-09T12:10:46.886382+00:00
None    19149018   15040169     Ivan Kogut   257122        Livyi Bereh      29    Yellow   Foul 2025-05-09T12:10:46.886382+00:00
None    19416785   22144895 Vegard Kongsro    10210      Yverdon Sport      22    Yellow   None 2025-05-09T12:10:47.069179+00:00

🔸 CHECKPOINT
------------------------------
 id  processed_count  penalties_count  cards_count                  timestamp
  1           152292            30228       457030 2025-05-09T16:22:08.103547

🔸 CITIES
------------------------------
 id  region_id          name                 updated_at                                              

"""
## Data Quality Assessment
Let's check for data quality issues like missing values, duplicates, and data distribution.
"""

In [10]:
def assess_data_quality(db_path, tables):
    conn = sqlite3.connect(db_path)
    
    print("🔍 DATA QUALITY ASSESSMENT")
    print("=" * 50)
    
    quality_report = {}
    
    for table in tables:
        print(f"\n📊 {table}")
        print("-" * 25)
        
        try:
            # Get all data for quality check (limit to avoid memory issues)
            df = pd.read_sql_query(f"SELECT * FROM {table} LIMIT 10000", conn)
            
            if len(df) == 0:
                print("  No data to analyze")
                continue
                
            # Basic statistics
            total_rows = len(df)
            total_cols = len(df.columns)
            
            print(f"  Rows analyzed: {total_rows:,}")
            print(f"  Columns: {total_cols}")
            
            # Check for missing values
            missing_data = df.isnull().sum()
            if missing_data.sum() > 0:
                print("  Missing values:")
                for col, missing_count in missing_data[missing_data > 0].items():
                    percentage = (missing_count / total_rows) * 100
                    print(f"    {col}: {missing_count} ({percentage:.1f}%)")
            else:
                print("  ✅ No missing values")
            
            # Check for potential duplicates (if table has reasonable size)
            if total_rows > 1 and total_rows <= 5000:
                duplicates = df.duplicated().sum()
                if duplicates > 0:
                    print(f"  ⚠️ Potential duplicate rows: {duplicates}")
                else:
                    print("  ✅ No duplicate rows detected")
            
            quality_report[table] = {
                'total_rows': total_rows,
                'missing_values': missing_data.to_dict(),
                'columns': list(df.columns)
            }
            
        except Exception as e:
            print(f"  Error analyzing {table}: {e}")
    
    conn.close()
    return quality_report

quality_report = assess_data_quality(db_path, tables)

🔍 DATA QUALITY ASSESSMENT

📊 bookmakers
-------------------------
  No data to analyze

📊 cards
-------------------------
  Rows analyzed: 10,000
  Columns: 10
  Missing values:
    id: 10000 (100.0%)
    player_id: 62 (0.6%)
    player_name: 62 (0.6%)
    reason: 728 (7.3%)

📊 checkpoint
-------------------------
  Rows analyzed: 1
  Columns: 5
  ✅ No missing values

📊 cities
-------------------------
  Rows analyzed: 10,000
  Columns: 5
  ✅ No missing values

📊 coaches
-------------------------
  Rows analyzed: 3,864
  Columns: 7
  Missing values:
    nationality: 135 (3.5%)
    birthdate: 422 (10.9%)
  ✅ No duplicate rows detected

📊 commentaries
-------------------------
  Rows analyzed: 10,000
  Columns: 6
  Missing values:
    minute: 217 (2.2%)

📊 continents
-------------------------
  Rows analyzed: 7
  Columns: 4
  ✅ No missing values
  ✅ No duplicate rows detected

📊 countries
-------------------------
  Rows analyzed: 242
  Columns: 6
  Missing values:
    iso2: 242 (100.0%)

## Database Relationships
Let's examine foreign key relationships between tables to understand how the data is connected.
"""

In [11]:
def analyze_relationships(db_path, tables):
    conn = sqlite3.connect(db_path)
    
    print("🔗 DATABASE RELATIONSHIPS")
    print("=" * 40)
    
    relationships = {}
    
    for table in tables:
        # Get foreign key information
        fk_df = pd.read_sql_query(f"PRAGMA foreign_key_list({table})", conn)
        
        if len(fk_df) > 0:
            print(f"\n📊 {table}")
            print("-" * 20)
            
            for _, fk in fk_df.iterrows():
                print(f"  {fk['from']} → {fk['table']}.{fk['to']}")
                
            relationships[table] = fk_df.to_dict('records')
        
    if not relationships:
        print("\n ℹ️ No explicit foreign key relationships found")
        print("   (This is common in SQLite databases)")
    
    conn.close()
    return relationships

relationships = analyze_relationships(db_path, tables)

🔗 DATABASE RELATIONSHIPS

📊 cities
--------------------
  region_id → regions.id

📊 commentaries
--------------------
  fixture_id → fixtures.id

📊 countries
--------------------
  continent_id → continents.id

📊 events
--------------------
  player_id → players.id
  team_id → teams.id
  fixture_id → fixtures.id

📊 expected_xg
--------------------
  team_id → teams.id
  fixture_id → fixtures.id

📊 fixtures
--------------------
  referee_id → referees.id
  venue_id → venues.id
  away_team_id → teams.id
  home_team_id → teams.id
  round_id → rounds.id
  stage_id → stages.id
  season_id → seasons.id
  league_id → leagues.id

📊 groups
--------------------
  season_id → seasons.id

📊 leagues
--------------------
  country_id → countries.id

📊 lineups
--------------------
  player_id → players.id
  team_id → teams.id
  fixture_id → fixtures.id

📊 markets
--------------------
  bookmaker_id → bookmakers.id

📊 news
--------------------
  league_id → leagues.id
  fixture_id → fixtures.id

📊 odd

## Summary Statistics
Let's create a summary view of our database with key metrics for each table.
"""

In [12]:
def create_summary_table(table_info, quality_report):
    print("📈 DATABASE SUMMARY")
    print("=" * 50)
    
    summary_data = []
    
    for table_name, info in table_info.items():
        row_count = info['rows']
        col_count = info['columns']
        
        # Get missing value info if available
        missing_info = ""
        if table_name in quality_report:
            missing_values = quality_report[table_name]['missing_values']
            total_missing = sum(missing_values.values())
            if total_missing > 0:
                missing_info = f"{total_missing:,} nulls"
            else:
                missing_info = "Complete"
        
        summary_data.append({
            'Table': table_name,
            'Rows': f"{row_count:,}",
            'Columns': col_count,
            'Data Quality': missing_info
        })
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
    
    return summary_df

summary_df = create_summary_table(table_info, quality_report)


📈 DATABASE SUMMARY
                  Table        Rows  Columns Data Quality
             bookmakers           0        4             
                  cards     459,278       10 10,852 nulls
             checkpoint           1        5     Complete
                 cities     102,231        5     Complete
                coaches       3,864        7    557 nulls
           commentaries   1,432,867        6    217 nulls
             continents           7        4     Complete
              countries         242        6    242 nulls
             evaluation           0       25             
                 events   1,779,888        9 29,381 nulls
            expected_xg           0        6             
           fixture_odds 104,236,537       11     Complete
     fixture_team_names       6,224        6     60 nulls
               fixtures     155,552       15 22,322 nulls
                 groups           0        5             
                leagues          27        7     27 n

## Next Steps
Based on our database analysis, we now have a comprehensive understanding of the Sportsmonks database structure. 

### Key Findings:
- Database contains [X] tables with sports-related data
- Main data types appear to be [describe based on table names]
- Data quality is [assess based on analysis]

# Football Match Outcome Prediction System
## Step 1: Data Exploration & Initial Setup

**Goal**: Build a machine learning system to predict football match outcomes for betting purposes.

**Models to implement**: XGBoost, SVM, Random Forest, Neural Networks

**Final deliverable**: Streamlit dashboard for live predictions

In this first step, we'll:
1. Set up our environment and database connection
2. Explore the available data for feature engineering
3. Understand the target variables we want to predict
4. Assess data completeness for our prediction timeframe
"""


In [14]:
# enviroment setup
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set display options for better data viewing
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✅ Libraries imported successfully!")
print(f"📅 Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Database connection
db_path = '/Users/sebastianvinther/Desktop/Sportsmonks/db_sportmonks.db'

def get_db_connection():
    """Create database connection with error handling"""
    try:
        conn = sqlite3.connect(db_path)
        return conn
    except Exception as e:
        print(f"❌ Database connection error: {e}")
        return None

# Test connection
conn = get_db_connection()
if conn:
    print("✅ Database connected successfully!")
    conn.close()
else:
    print("❌ Failed to connect to database")


✅ Libraries imported successfully!
📅 Analysis date: 2025-05-26 19:07:47
✅ Database connected successfully!


## 🔍 Understanding Our Target Variable

For betting predictions, we need to understand what we're predicting:
- **Primary target**: Match outcome (Home Win, Draw, Away Win)
- **Secondary targets**: Over/Under 2.5 goals, Both teams to score

Let's examine the fixtures table to understand our target data.
"""

In [15]:
# CODE: Target Variable Analysis
def analyze_match_outcomes():
    """Analyze the distribution of match outcomes in our dataset"""
    conn = get_db_connection()
    
    print("🎯 ANALYZING MATCH OUTCOMES")
    print("=" * 50)
    
    # Get fixtures with complete results
    query = """
    SELECT 
        COUNT(*) as total_matches,
        COUNT(CASE WHEN score_home IS NOT NULL AND score_away IS NOT NULL THEN 1 END) as matches_with_scores,
        COUNT(CASE WHEN score_home > score_away THEN 1 END) as home_wins,
        COUNT(CASE WHEN score_home = score_away THEN 1 END) as draws,
        COUNT(CASE WHEN score_home < score_away THEN 1 END) as away_wins,
        COUNT(CASE WHEN (score_home + score_away) > 2.5 THEN 1 END) as over_25_goals,
        COUNT(CASE WHEN score_home > 0 AND score_away > 0 THEN 1 END) as both_teams_scored
    FROM fixtures 
    WHERE score_home IS NOT NULL AND score_away IS NOT NULL
    """
    
    results = pd.read_sql_query(query, conn)
    
    total_with_scores = results['matches_with_scores'].iloc[0]
    home_wins = results['home_wins'].iloc[0]
    draws = results['draws'].iloc[0] 
    away_wins = results['away_wins'].iloc[0]
    over_25 = results['over_25_goals'].iloc[0]
    btts = results['both_teams_scored'].iloc[0]
    
    print(f"📊 Total matches with results: {total_with_scores:,}")
    print(f"🏠 Home wins: {home_wins:,} ({home_wins/total_with_scores*100:.1f}%)")
    print(f"🤝 Draws: {draws:,} ({draws/total_with_scores*100:.1f}%)")
    print(f"✈️ Away wins: {away_wins:,} ({away_wins/total_with_scores*100:.1f}%)")
    print(f"⚽ Over 2.5 goals: {over_25:,} ({over_25/total_with_scores*100:.1f}%)")
    print(f"🎯 Both teams scored: {btts:,} ({btts/total_with_scores*100:.1f}%)")
    
    # Check data completeness over time
    print(f"\n📅 TEMPORAL DATA DISTRIBUTION")
    print("-" * 30)
    
    temporal_query = """
    SELECT 
        substr(starting_at, 1, 4) as year,
        COUNT(*) as matches,
        COUNT(CASE WHEN score_home IS NOT NULL THEN 1 END) as with_results
    FROM fixtures 
    WHERE starting_at IS NOT NULL
    GROUP BY substr(starting_at, 1, 4)
    ORDER BY year DESC
    """
    
    temporal_df = pd.read_sql_query(temporal_query, conn)
    print(temporal_df.to_string(index=False))
    
    conn.close()
    return results, temporal_df

outcome_stats, temporal_data = analyze_match_outcomes()


🎯 ANALYZING MATCH OUTCOMES
📊 Total matches with results: 154,495
🏠 Home wins: 69,710 (45.1%)
🤝 Draws: 38,931 (25.2%)
✈️ Away wins: 45,854 (29.7%)
⚽ Over 2.5 goals: 79,006 (51.1%)
🎯 Both teams scored: 80,523 (52.1%)

📅 TEMPORAL DATA DISTRIBUTION
------------------------------
year  matches  with_results
2025     3737          3083
2024     7867          7867
2023     8158          8156
2022     7613          7516
2021     8467          8463
2020     7117          6900
2019     7902          7900
2018     7832          7830
2017     8100          8099
2016     7884          7883
2015     8004          8000
2014     7762          7746
2013     7789          7780
2012     7985          7966
2011     7454          7453
2010     7453          7453
2009     7170          7170
2008     7332          7332
2007     6922          6922
2006     6931          6903
2005     4115          4115
2004      630           630
2003      351           351
2002      392           392
2001      371           

## 🏆 League and Competition Analysis

Different leagues have different characteristics (home advantage, scoring patterns, etc.).
Let's understand which leagues we have data for and their quality.
"""

In [17]:
#  CODE: League Analysis
def analyze_leagues():
    """Analyze available leagues and their data quality"""
    conn = get_db_connection()
    
    print("🏆 LEAGUE ANALYSIS")
    print("=" * 40)
    
    league_query = """
    SELECT 
        l.name as league_name,
        l.country_id,
        c.name as country,
        COUNT(f.id) as total_fixtures,
        COUNT(CASE WHEN f.score_home IS NOT NULL THEN 1 END) as completed_fixtures,
        MIN(f.starting_at) as first_match,
        MAX(f.starting_at) as last_match,
        COUNT(DISTINCT f.season_id) as seasons
    FROM leagues l
    LEFT JOIN countries c ON l.country_id = c.id
    LEFT JOIN fixtures f ON l.id = f.league_id
    GROUP BY l.id, l.name, c.name
    HAVING total_fixtures > 0
    ORDER BY completed_fixtures DESC
    """
    
    leagues_df = pd.read_sql_query(league_query, conn)
    
    print("📊 Top leagues by data volume:")
    print(leagues_df.head(10).to_string(index=False))
    
    # Calculate completion rates
    leagues_df['completion_rate'] = (leagues_df['completed_fixtures'] / leagues_df['total_fixtures'] * 100).round(1)
    
    print(f"\n🎯 Best data quality leagues (>90% completion):")
    quality_leagues = leagues_df[leagues_df['completion_rate'] > 90].head(5)
    print(quality_leagues[['league_name', 'country', 'completed_fixtures', 'completion_rate']].to_string(index=False))
    
    conn.close()
    return leagues_df

leagues_analysis = analyze_leagues()


🏆 LEAGUE ANALYSIS
📊 Top leagues by data volume:
   league_name  country_id     country  total_fixtures  completed_fixtures         first_match          last_match  seasons
        FA Cup         462     England           13521               13466 2005-11-03 23:00:00 2025-05-17 15:30:00       20
  Championship         462     England           11138               11137 2005-08-05 22:00:00 2025-05-24 14:01:00       20
Premier League         462     England            9500                9484 2000-08-18 22:00:00 2025-05-25 15:00:00       25
     La Liga 2          32       Spain            9303                9267 2005-08-26 22:00:00 2025-06-01 00:00:00       20
       Serie B         251       Italy            8741                8734 2005-08-25 22:00:00 2025-06-01 00:00:00       20
       Serie A         251       Italy            7601                7577 2005-08-27 16:00:00 2025-05-25 13:00:00       20
       La Liga          32       Spain            7600                7573 2005-08-2

## 📊 Available Features Analysis

For accurate predictions, we need to understand what features are available:
- **Team statistics**: Goals, shots, possession, etc.
- **Player statistics**: Goals, assists, cards
- **Historical data**: Head-to-head, recent form
- **Match context**: Home/away, league, season

Let's examine the statistics tables to see what features we can engineer.
"""

In [18]:
# CODE: Feature Availability Analysis
def analyze_available_features():
    """Analyze what statistics and features are available for modeling"""
    conn = get_db_connection()
    
    print("📊 AVAILABLE FEATURES ANALYSIS")
    print("=" * 50)
    
    # Team statistics types
    print("🏆 Team Statistics Types:")
    team_stats_query = """
    SELECT 
        type,
        COUNT(*) as frequency,
        COUNT(DISTINCT fixture_id) as unique_fixtures
    FROM statistics 
    WHERE type IS NOT NULL
    GROUP BY type
    ORDER BY frequency DESC
    """
    
    team_stats = pd.read_sql_query(team_stats_query, conn)
    print(team_stats.head(15).to_string(index=False))
    
    print(f"\n👤 Player Statistics Types:")
    player_stats_query = """
    SELECT 
        type,
        COUNT(*) as frequency,
        COUNT(DISTINCT fixture_id) as unique_fixtures,
        COUNT(DISTINCT player_id) as unique_players
    FROM player_statistics 
    WHERE type IS NOT NULL
    GROUP BY type
    ORDER BY frequency DESC
    """
    
    player_stats = pd.read_sql_query(player_stats_query, conn)
    print(player_stats.head(15).to_string(index=False))
    
    # Check data coverage for recent matches
    print(f"\n📅 Recent Data Coverage (last 1000 completed fixtures):")
    coverage_query = """
    SELECT 
        COUNT(DISTINCT f.id) as fixtures,
        COUNT(DISTINCT s.fixture_id) as with_team_stats,
        COUNT(DISTINCT ps.fixture_id) as with_player_stats,
        COUNT(DISTINCT c.fixture_id) as with_cards,
        COUNT(DISTINCT l.fixture_id) as with_lineups
    FROM (
        SELECT id FROM fixtures 
        WHERE score_home IS NOT NULL 
        ORDER BY starting_at DESC 
        LIMIT 1000
    ) f
    LEFT JOIN statistics s ON f.id = s.fixture_id
    LEFT JOIN player_statistics ps ON f.id = ps.fixture_id
    LEFT JOIN cards c ON f.id = c.fixture_id
    LEFT JOIN lineups l ON f.id = l.fixture_id
    """
    
    coverage = pd.read_sql_query(coverage_query, conn)
    print(coverage.to_string(index=False))
    
    conn.close()
    return team_stats, player_stats, coverage

team_stats, player_stats, data_coverage = analyze_available_features()


📊 AVAILABLE FEATURES ANALYSIS
🏆 Team Statistics Types:
                          type  frequency  unique_fixtures
                          None      82367            82367
                       Corners      79024            79024
             Ball Possession %      77732            77732
                         Goals      72369            72369
                   Yellowcards      66562            66562
                      Redcards      57215            57215
Successful Dribbles Percentage      47726            47726
               Yellowred Cards      33096            33096
                       Assists      22245            22245

👤 Player Statistics Types:
                      type  frequency  unique_fixtures  unique_players
            Minutes Played    3133666           113603           51569
            Goals Conceded    2353714            97267           55012
                    Passes    1656462            57404           32035
           Accurate Passes    1640064      

## 🎲 Betting Data Analysis

Since our goal is betting predictions, let's analyze the available odds data to understand:
- Which bookmakers and markets we have
- Odds coverage for recent matches
- Potential for value betting analysis
"""

In [19]:
#CELL 10 - CODE: Betting Data Analysis
def analyze_betting_data():
    """Analyze available betting odds and markets"""
    conn = get_db_connection()
    
    print("🎲 BETTING DATA ANALYSIS")
    print("=" * 40)
    
    # Overall odds coverage
    odds_overview_query = """
    SELECT 
        COUNT(DISTINCT fixture_id) as fixtures_with_odds,
        COUNT(DISTINCT bookmaker_name) as unique_bookmakers,
        COUNT(DISTINCT market_name) as unique_markets,
        COUNT(*) as total_odds_records
    FROM fixture_odds
    """
    
    odds_overview = pd.read_sql_query(odds_overview_query, conn)
    print("📊 Odds Data Overview:")
    print(odds_overview.to_string(index=False))
    
    # Top bookmakers by coverage
    print(f"\n🏪 Top Bookmakers by Coverage:")
    bookmaker_query = """
    SELECT 
        bookmaker_name,
        COUNT(DISTINCT fixture_id) as fixtures_covered,
        COUNT(*) as total_odds
    FROM fixture_odds
    WHERE bookmaker_name IS NOT NULL
    GROUP BY bookmaker_name
    ORDER BY fixtures_covered DESC
    LIMIT 10
    """
    
    bookmakers = pd.read_sql_query(bookmaker_query, conn)
    print(bookmakers.to_string(index=False))
    
    # Available markets
    print(f"\n📈 Available Betting Markets:")
    markets_query = """
    SELECT 
        market_name,
        COUNT(DISTINCT fixture_id) as fixtures_covered,
        COUNT(*) as total_odds
    FROM fixture_odds
    WHERE market_name IS NOT NULL
    GROUP BY market_name
    ORDER BY fixtures_covered DESC
    LIMIT 15
    """
    
    markets = pd.read_sql_query(markets_query, conn)
    print(markets.to_string(index=False))
    
    # Recent odds coverage
    print(f"\n📅 Recent Odds Coverage (last 30 days of data):")
    recent_coverage_query = """
    SELECT 
        COUNT(DISTINCT f.id) as recent_fixtures,
        COUNT(DISTINCT fo.fixture_id) as fixtures_with_odds,
        ROUND(COUNT(DISTINCT fo.fixture_id) * 100.0 / COUNT(DISTINCT f.id), 1) as coverage_percentage
    FROM fixtures f
    LEFT JOIN fixture_odds fo ON f.id = fo.fixture_id
    WHERE f.starting_at >= date('now', '-30 days')
    AND f.score_home IS NOT NULL
    """
    
    recent_coverage = pd.read_sql_query(recent_coverage_query, conn)
    print(recent_coverage.to_string(index=False))
    
    conn.close()
    return odds_overview, bookmakers, markets, recent_coverage

odds_overview, bookmakers, markets, recent_coverage = analyze_betting_data()


🎲 BETTING DATA ANALYSIS
📊 Odds Data Overview:
 fixtures_with_odds  unique_bookmakers  unique_markets  total_odds_records
              55667                 26             186           104236537

🏪 Top Bookmakers by Coverage:
   bookmaker_name  fixtures_covered  total_odds
Unknown Bookmaker             55268    49052482
         Pinnacle             54566     3829690
          Dafabet             54365      650214
      Marathonbet             53259     2247008
            10Bet             53234     5831756
           188Bet             52420     1114878
           Unibet             50884     5361014
           bet365             50781    18230351
          Betfair             50778     2930475
         888Sport             49306     3780724

📈 Available Betting Markets:
        market_name  fixtures_covered  total_odds
    Fulltime Result             55590     4784414
     Asian Handicap             55567     2000832
   Goals Over/Under             55563     2601091
      Double Ch

# 📋 Step 1 Summary & Next Steps

### ✅ What we've discovered:

**Data Quality:**
- We have substantial match data with good completion rates
- Multiple leagues with varying data quality
- Rich statistical data for both teams and players
- Comprehensive betting odds from multiple bookmakers

**Target Variables:**
- Primary: Match outcome (Home/Draw/Away)
- Secondary: Over/Under 2.5 goals, Both teams to score
- Good class distribution for machine learning

**Feature Engineering Opportunities:**
- Team form metrics (recent performance)
- Head-to-head records
- Player availability and form
- League-specific patterns
- Betting market analysis

### 🎯 Next Step: Feature Engineering


In Step 2, we'll create the features needed for our machine learning models:
1. **Team Form Features**: Last 5-10 games performance
2. **Historical Features**: Head-to-head records, venue performance
3. **Statistical Features**: Average goals, shots, possession, etc.
4. **Player Features**: Key player availability and form
5. **Contextual Features**: League difficulty, seasonality

# 🔬 Step 2: Feature Engineering for Machine Learning

**Goal**: Create predictive features from our rich Sportsmonks database

**What we'll build:**
1. **Team Form Features** - Recent performance metrics (last 5-10 games)
2. **Head-to-Head Features** - Historical matchup records
3. **Statistical Features** - Goals, possession, cards averages
4. **Player Strength Features** - Key player availability and ratings
5. **Match Context Features** - Home advantage, league difficulty

**Output**: Clean training dataset ready for XGBoost, SVM, Random Forest, Neural Networks

From Step 1, we discovered:
- ✅ 155k+ fixtures with results for training
- ✅ Rich team statistics (goals, possession, corners, cards)
- ✅ Comprehensive player data (3M+ records, ratings, passes)
- ✅ 55k+ fixtures with betting odds for validation
"""

this could also be done thorugh a new file, so it would just import it from this file, and call the new for feature engineering.

In [20]:
# CELL 2 - CODE: Setup and Helper Functions
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Database connection
db_path = '/Users/sebastianvinther/Desktop/Sportsmonks/db_sportmonks.db'

def get_db_connection():
    return sqlite3.connect(db_path)

def safe_divide(a, b, default=0):
    """Safe division with default value for zero division"""
    return np.where(b != 0, a / b, default)

def calculate_days_between(date1, date2):
    """Calculate days between two dates"""
    try:
        d1 = pd.to_datetime(date1)
        d2 = pd.to_datetime(date2)
        return (d1 - d2).dt.days
    except:
        return 0

print("🔧 Setup completed!")
print("📅 Starting feature engineering...")


🔧 Setup completed!
📅 Starting feature engineering...


"""
## 🎯 Step 2.1: Core Match Dataset

First, let's create our base dataset with all completed fixtures that have the data we need.
We'll focus on matches from the last 3 years for relevance and data quality.
"""

In [21]:
# CELL 4 - CODE: Create Base Match Dataset
def create_base_match_dataset():
    """Create the foundation dataset with completed matches"""
    conn = get_db_connection()
    
    print("🏗️ CREATING BASE MATCH DATASET")
    print("=" * 50)
    
    # Get completed fixtures with team information
    base_query = """
    SELECT 
        f.id as fixture_id,
        f.league_id,
        f.season_id,
        f.home_team_id,
        f.away_team_id,
        f.starting_at as match_date,
        f.score_home,
        f.score_away,
        l.name as league_name,
        l.country_id,
        c.name as country_name,
        ht.name as home_team_name,
        at.name as away_team_name,
        s.name as season_name
    FROM fixtures f
    JOIN leagues l ON f.league_id = l.id
    JOIN countries c ON l.country_id = c.id
    JOIN teams ht ON f.home_team_id = ht.id
    JOIN teams at ON f.away_team_id = at.id
    JOIN seasons s ON f.season_id = s.id
    WHERE f.score_home IS NOT NULL 
    AND f.score_away IS NOT NULL
    AND f.starting_at IS NOT NULL
    AND f.starting_at >= date('now', '-3 years')
    ORDER BY f.starting_at DESC
    """
    
    base_df = pd.read_sql_query(base_query, conn)
    
    # Create target variables
    base_df['total_goals'] = base_df['score_home'] + base_df['score_away']
    base_df['goal_difference'] = base_df['score_home'] - base_df['score_away']
    
    # Match outcome (our main target)
    base_df['result'] = np.where(base_df['score_home'] > base_df['score_away'], 'H',
                        np.where(base_df['score_home'] < base_df['score_away'], 'A', 'D'))
    
    # Additional targets for betting
    base_df['over_2_5'] = (base_df['total_goals'] > 2.5).astype(int)
    base_df['btts'] = ((base_df['score_home'] > 0) & (base_df['score_away'] > 0)).astype(int)
    base_df['home_win'] = (base_df['result'] == 'H').astype(int)
    base_df['draw'] = (base_df['result'] == 'D').astype(int)
    base_df['away_win'] = (base_df['result'] == 'A').astype(int)
    
    # Convert match_date to datetime
    base_df['match_date'] = pd.to_datetime(base_df['match_date'])
    
    print(f"📊 Base dataset created:")
    print(f"   Total matches: {len(base_df):,}")
    print(f"   Date range: {base_df['match_date'].min()} to {base_df['match_date'].max()}")
    print(f"   Leagues: {base_df['league_name'].nunique()}")
    print(f"   Teams: {base_df[['home_team_id', 'away_team_id']].stack().nunique()}")
    
    print(f"\n🎯 Target distribution:")
    print(f"   Home wins: {base_df['home_win'].mean():.3f}")
    print(f"   Draws: {base_df['draw'].mean():.3f}")
    print(f"   Away wins: {base_df['away_win'].mean():.3f}")
    print(f"   Over 2.5 goals: {base_df['over_2_5'].mean():.3f}")
    print(f"   Both teams score: {base_df['btts'].mean():.3f}")
    
    conn.close()
    return base_df

base_matches = create_base_match_dataset()


🏗️ CREATING BASE MATCH DATASET
📊 Base dataset created:
   Total matches: 22,774
   Date range: 2022-05-26 16:00:00 to 2025-06-01 00:00:00
   Leagues: 24
   Teams: 1407

🎯 Target distribution:
   Home wins: 0.441
   Draws: 0.246
   Away wins: 0.313
   Over 2.5 goals: 0.532
   Both teams score: 0.533


## ⚽ Step 2.2: Team Form Features

Now we'll calculate team form based on recent performance. This is crucial for predictions as recent form often matters more than historical averages.

**Form Features:**
- Goals scored/conceded per game (last 5 & 10 games)
- Win/draw/loss rates
- Home/away specific form
- Goals timing patterns
"""

In [23]:
def calculate_team_form_features(base_df, form_games=[5, 10]):
    """Calculate team form features for recent games"""
    conn = get_db_connection()
    
    print("⚽ CALCULATING TEAM FORM FEATURES")
    print("=" * 50)
    
    # We'll build form features for each team
    form_features = []
    
    # Get all unique teams
    all_teams = pd.concat([
        base_df[['home_team_id', 'home_team_name']].rename(columns={'home_team_id': 'team_id', 'home_team_name': 'team_name'}),
        base_df[['away_team_id', 'away_team_name']].rename(columns={'away_team_id': 'team_id', 'away_team_name': 'team_name'})
    ]).drop_duplicates()
    
    print(f"📊 Processing form for {len(all_teams)} teams...")
    
    for idx, (_, match) in enumerate(base_matches.iterrows()):
        if idx % 1000 == 0:
            print(f"   Processed {idx:,} matches...")
        
        match_date = match['match_date']
        home_team = match['home_team_id']
        away_team = match['away_team_id']
        
        # Calculate form for both teams
        for team_id, is_home in [(home_team, True), (away_team, False)]:
            
            # Get recent matches for this team before current match
            team_recent = base_df[
                ((base_df['home_team_id'] == team_id) | (base_df['away_team_id'] == team_id)) &
                (base_df['match_date'] < match_date)
            ].sort_values('match_date', ascending=False)
            
            form_dict = {
                'fixture_id': match['fixture_id'],
                'team_id': team_id,
                'is_home': is_home
            }
            
            # Calculate form for different game windows
            for games in form_games:
                recent_games = team_recent.head(games)
                
                if len(recent_games) >= min(3, games):  # Need minimum games for reliable form
                    
                    # Goals for/against calculation
                    goals_for = []
                    goals_against = []
                    results = []
                    home_games = 0
                    
                    for _, game in recent_games.iterrows():
                        if game['home_team_id'] == team_id:  # Team was home
                            goals_for.append(game['score_home'])
                            goals_against.append(game['score_away'])
                            if game['score_home'] > game['score_away']:
                                results.append('W')
                            elif game['score_home'] < game['score_away']:
                                results.append('L')
                            else:
                                results.append('D')
                            home_games += 1
                        else:  # Team was away
                            goals_for.append(game['score_away'])
                            goals_against.append(game['score_home'])
                            if game['score_away'] > game['score_home']:
                                results.append('W')
                            elif game['score_away'] < game['score_home']:
                                results.append('L')
                            else:
                                results.append('D')
                    
                    # Form metrics
                    form_dict[f'form_{games}_games_played'] = len(recent_games)
                    form_dict[f'form_{games}_goals_for_avg'] = np.mean(goals_for) if goals_for else 0
                    form_dict[f'form_{games}_goals_against_avg'] = np.mean(goals_against) if goals_against else 0
                    form_dict[f'form_{games}_goal_diff_avg'] = np.mean(goals_for) - np.mean(goals_against) if goals_for else 0
                    form_dict[f'form_{games}_wins'] = results.count('W')
                    form_dict[f'form_{games}_draws'] = results.count('D')
                    form_dict[f'form_{games}_losses'] = results.count('L')
                    form_dict[f'form_{games}_win_rate'] = results.count('W') / len(results) if results else 0
                    form_dict[f'form_{games}_points_per_game'] = (results.count('W') * 3 + results.count('D')) / len(results) if results else 0
                    form_dict[f'form_{games}_home_games_pct'] = home_games / len(recent_games) if len(recent_games) > 0 else 0.5
                    
                else:
                    # Not enough games for reliable form
                    for metric in ['games_played', 'goals_for_avg', 'goals_against_avg', 'goal_diff_avg', 
                                  'wins', 'draws', 'losses', 'win_rate', 'points_per_game', 'home_games_pct']:
                        form_dict[f'form_{games}_{metric}'] = 0 if 'avg' not in metric and 'rate' not in metric and 'pct' not in metric else 0.0
            
            form_features.append(form_dict)
    
    form_df = pd.DataFrame(form_features)
    
    print(f"✅ Team form features calculated!")
    print(f"   Features per team: {len([col for col in form_df.columns if col.startswith('form_')])}")
    print(f"   Total records: {len(form_df):,}")
    
    return form_df

# Calculate form features (this will take a few minutes due to the complexity)
print("🕐 Calculating team form features (this may take 3-5 minutes)...")
team_form = calculate_team_form_features(base_matches)

🕐 Calculating team form features (this may take 3-5 minutes)...
⚽ CALCULATING TEAM FORM FEATURES
📊 Processing form for 1407 teams...
   Processed 0 matches...
   Processed 1,000 matches...
   Processed 2,000 matches...
   Processed 3,000 matches...
   Processed 4,000 matches...
   Processed 5,000 matches...
   Processed 6,000 matches...
   Processed 7,000 matches...
   Processed 8,000 matches...
   Processed 9,000 matches...
   Processed 10,000 matches...
   Processed 11,000 matches...
   Processed 12,000 matches...
   Processed 13,000 matches...
   Processed 14,000 matches...
   Processed 15,000 matches...
   Processed 16,000 matches...
   Processed 17,000 matches...
   Processed 18,000 matches...
   Processed 19,000 matches...
   Processed 20,000 matches...
   Processed 21,000 matches...
   Processed 22,000 matches...
✅ Team form features calculated!
   Features per team: 20
   Total records: 45,548


## 📊 Step 2.3: Team Statistical Features

Let's extract the rich statistical data we identified in Step 1 to create features about teams' playing styles and strengths.

**Statistical Features:**
- Average possession, shots, corners
- Defensive metrics (cards, fouls)
- Playing style indicators
- League-adjusted statistics
"""

In [24]:
# CELL 8 - CODE: Extract Team Statistical Features
def extract_team_statistical_features(base_df):
    """Extract team statistical features from the statistics table"""
    conn = get_db_connection()
    
    print("📊 EXTRACTING TEAM STATISTICAL FEATURES")
    print("=" * 50)
    
    # Get team statistics for recent matches
    stats_query = """
    SELECT 
        s.fixture_id,
        s.team_id,
        s.type,
        CAST(s.value AS REAL) as stat_value
    FROM statistics s
    WHERE s.fixture_id IN ({})
    AND s.type IS NOT NULL
    AND s.value IS NOT NULL
    AND s.value != ''
    """.format(','.join(map(str, base_df['fixture_id'].unique())))
    
    print("🔄 Extracting statistics from database...")
    team_stats_raw = pd.read_sql_query(stats_query, conn)
    
    print(f"📈 Raw statistics extracted: {len(team_stats_raw):,} records")
    print(f"   Stat types: {team_stats_raw['type'].nunique()}")
    
    # Pivot to get one row per team per match
    team_stats_pivot = team_stats_raw.pivot_table(
        index=['fixture_id', 'team_id'], 
        columns='type', 
        values='stat_value', 
        aggfunc='first'
    ).reset_index()
    
    # Clean column names
    team_stats_pivot.columns = ['fixture_id', 'team_id'] + [f'stat_{col.lower().replace(" ", "_").replace("%", "pct")}' for col in team_stats_pivot.columns[2:]]
    
    # Fill missing values with 0 for numeric stats
    stat_columns = [col for col in team_stats_pivot.columns if col.startswith('stat_')]
    team_stats_pivot[stat_columns] = team_stats_pivot[stat_columns].fillna(0)
    
    print(f"✅ Team statistical features extracted!")
    print(f"   Features per team: {len(stat_columns)}")
    print(f"   Top statistics:")
    
    # Show most common statistics
    coverage = team_stats_pivot[stat_columns].notna().mean().sort_values(ascending=False)
    for stat, cov in coverage.head(10).items():
        print(f"      {stat}: {cov:.1%} coverage")
    
    conn.close()
    return team_stats_pivot

team_stats = extract_team_statistical_features(base_matches)

📊 EXTRACTING TEAM STATISTICAL FEATURES
🔄 Extracting statistics from database...
📈 Raw statistics extracted: 168,046 records
   Stat types: 9
✅ Team statistical features extracted!
   Features per team: 9
   Top statistics:
      stat_assists: 100.0% coverage
      stat_ball_possession_pct: 100.0% coverage
      stat_corners: 100.0% coverage
      stat_goals: 100.0% coverage
      stat_none: 100.0% coverage
      stat_redcards: 100.0% coverage
      stat_successful_dribbles_percentage: 100.0% coverage
      stat_yellowcards: 100.0% coverage
      stat_yellowred_cards: 100.0% coverage


## 🤝 Step 2.4: Head-to-Head Features

Historical matchups between teams can be very predictive. Let's calculate head-to-head records.

**H2H Features:**
- Win/loss record between teams
- Goals scored/conceded patterns
- Recent H2H form
- Home/away H2H splits
"""

In [25]:
# CELL 10 - CODE: Calculate Head-to-Head Features
def calculate_head_to_head_features(base_df):
    """Calculate head-to-head features between teams"""
    
    print("🤝 CALCULATING HEAD-TO-HEAD FEATURES")
    print("=" * 50)
    
    h2h_features = []
    
    for idx, (_, match) in enumerate(base_df.iterrows()):
        if idx % 1000 == 0:
            print(f"   Processed {idx:,} matches...")
        
        home_team = match['home_team_id']
        away_team = match['away_team_id']
        match_date = match['match_date']
        
        # Get historical matches between these teams
        h2h_matches = base_df[
            (((base_df['home_team_id'] == home_team) & (base_df['away_team_id'] == away_team)) |
             ((base_df['home_team_id'] == away_team) & (base_df['away_team_id'] == home_team))) &
            (base_df['match_date'] < match_date)
        ].sort_values('match_date', ascending=False)
        
        h2h_dict = {
            'fixture_id': match['fixture_id'],
            'home_team_id': home_team,
            'away_team_id': away_team
        }
        
        if len(h2h_matches) >= 1:
            # Overall H2H record (from home team perspective)
            home_wins = 0
            draws = 0
            away_wins = 0
            total_home_goals = 0
            total_away_goals = 0
            
            for _, h2h in h2h_matches.iterrows():
                if h2h['home_team_id'] == home_team:  # Same fixture setup
                    total_home_goals += h2h['score_home']
                    total_away_goals += h2h['score_away']
                    if h2h['score_home'] > h2h['score_away']:
                        home_wins += 1
                    elif h2h['score_home'] < h2h['score_away']:
                        away_wins += 1
                    else:
                        draws += 1
                else:  # Reversed fixture setup
                    total_home_goals += h2h['score_away']
                    total_away_goals += h2h['score_home']
                    if h2h['score_away'] > h2h['score_home']:
                        home_wins += 1
                    elif h2h['score_away'] < h2h['score_home']:
                        away_wins += 1
                    else:
                        draws += 1
            
            total_games = len(h2h_matches)
            h2h_dict['h2h_games_played'] = total_games
            h2h_dict['h2h_home_wins'] = home_wins
            h2h_dict['h2h_draws'] = draws
            h2h_dict['h2h_away_wins'] = away_wins
            h2h_dict['h2h_home_win_rate'] = home_wins / total_games
            h2h_dict['h2h_draw_rate'] = draws / total_games
            h2h_dict['h2h_away_win_rate'] = away_wins / total_games
            h2h_dict['h2h_avg_home_goals'] = total_home_goals / total_games
            h2h_dict['h2h_avg_away_goals'] = total_away_goals / total_games
            h2h_dict['h2h_avg_total_goals'] = (total_home_goals + total_away_goals) / total_games
            
            # Recent H2H form (last 5 meetings)
            recent_h2h = h2h_matches.head(5)
            if len(recent_h2h) >= 1:
                recent_home_wins = 0
                recent_draws = 0
                recent_away_wins = 0
                
                for _, recent in recent_h2h.iterrows():
                    if recent['home_team_id'] == home_team:
                        if recent['score_home'] > recent['score_away']:
                            recent_home_wins += 1
                        elif recent['score_home'] < recent['score_away']:
                            recent_away_wins += 1
                        else:
                            recent_draws += 1
                    else:
                        if recent['score_away'] > recent['score_home']:
                            recent_home_wins += 1
                        elif recent['score_away'] < recent['score_home']:
                            recent_away_wins += 1
                        else:
                            recent_draws += 1
                
                h2h_dict['h2h_recent_games'] = len(recent_h2h)
                h2h_dict['h2h_recent_home_wins'] = recent_home_wins
                h2h_dict['h2h_recent_draws'] = recent_draws
                h2h_dict['h2h_recent_away_wins'] = recent_away_wins
            else:
                h2h_dict['h2h_recent_games'] = 0
                h2h_dict['h2h_recent_home_wins'] = 0
                h2h_dict['h2h_recent_draws'] = 0
                h2h_dict['h2h_recent_away_wins'] = 0
                
        else:
            # No historical matches
            for col in ['h2h_games_played', 'h2h_home_wins', 'h2h_draws', 'h2h_away_wins',
                       'h2h_recent_games', 'h2h_recent_home_wins', 'h2h_recent_draws', 'h2h_recent_away_wins']:
                h2h_dict[col] = 0
            for col in ['h2h_home_win_rate', 'h2h_draw_rate', 'h2h_away_win_rate',
                       'h2h_avg_home_goals', 'h2h_avg_away_goals', 'h2h_avg_total_goals']:
                h2h_dict[col] = 0.0
        
        h2h_features.append(h2h_dict)
    
    h2h_df = pd.DataFrame(h2h_features)
    
    print(f"✅ Head-to-head features calculated!")
    print(f"   Features per match: {len([col for col in h2h_df.columns if col.startswith('h2h_')])}")
    print(f"   Matches with H2H history: {(h2h_df['h2h_games_played'] > 0).sum():,}")
    
    return h2h_df

h2h_features = calculate_head_to_head_features(base_matches)


🤝 CALCULATING HEAD-TO-HEAD FEATURES
   Processed 0 matches...
   Processed 1,000 matches...
   Processed 2,000 matches...
   Processed 3,000 matches...
   Processed 4,000 matches...
   Processed 5,000 matches...
   Processed 6,000 matches...
   Processed 7,000 matches...
   Processed 8,000 matches...
   Processed 9,000 matches...
   Processed 10,000 matches...
   Processed 11,000 matches...
   Processed 12,000 matches...
   Processed 13,000 matches...
   Processed 14,000 matches...
   Processed 15,000 matches...
   Processed 16,000 matches...
   Processed 17,000 matches...
   Processed 18,000 matches...
   Processed 19,000 matches...
   Processed 20,000 matches...
   Processed 21,000 matches...
   Processed 22,000 matches...
✅ Head-to-head features calculated!
   Features per match: 14
   Matches with H2H history: 15,305


## 🏆 Step 2.5: League & Context Features

Different leagues have different characteristics. Let's create features that capture league difficulty, home advantage, and seasonal patterns.

**Context Features:**
- League strength/difficulty ratings
- Home advantage by league
- Seasonal patterns (early/mid/late season)
- Day of week effects
"""

In [26]:
# CELL 12 - CODE: Calculate League and Context Features
def calculate_context_features(base_df):
    """Calculate league and contextual features"""
    
    print("🏆 CALCULATING LEAGUE & CONTEXT FEATURES")
    print("=" * 50)
    
    context_features = base_df[['fixture_id', 'league_id', 'season_id', 'match_date', 'home_team_id', 'away_team_id']].copy()
    
    # League-level statistics
    league_stats = base_df.groupby('league_id').agg({
        'total_goals': 'mean',
        'over_2_5': 'mean',
        'btts': 'mean',
        'home_win': 'mean',
        'draw': 'mean',
        'away_win': 'mean'
    }).add_prefix('league_avg_')
    
    # Season-level statistics  
    season_stats = base_df.groupby(['league_id', 'season_id']).agg({
        'total_goals': 'mean',
        'home_win': 'mean'
    }).add_prefix('season_avg_')
    
    # Merge league stats
    context_features = context_features.merge(
        league_stats.reset_index(), 
        on='league_id', 
        how='left'
    )
    
    # Merge season stats
    context_features = context_features.merge(
        season_stats.reset_index(), 
        on=['league_id', 'season_id'], 
        how='left'
    )
    
    # Date-based features
    context_features['match_date'] = pd.to_datetime(context_features['match_date'])
    context_features['day_of_week'] = context_features['match_date'].dt.dayofweek
    context_features['month'] = context_features['match_date'].dt.month
    context_features['is_weekend'] = context_features['day_of_week'].isin([5, 6]).astype(int)
    
    # Season progress (approximate)
    context_features['season_month'] = context_features['month'].apply(
        lambda x: x if x >= 8 else x + 12  # Football season typically Aug-May
    )
    context_features['season_progress'] = (context_features['season_month'] - 8) / 9  # 0-1 scale
    
    print(f"✅ Context features calculated!")
    print(f"   League features: {len([col for col in context_features.columns if 'league_' in col])}")
    print(f"   Temporal features: {len([col for col in context_features.columns if any(x in col for x in ['day_', 'month', 'season_', 'weekend'])])}")
    
    return context_features

context_features = calculate_context_features(base_matches)

🏆 CALCULATING LEAGUE & CONTEXT FEATURES
✅ Context features calculated!
   League features: 7
   Temporal features: 8


## 🔗 Step 2.6: Combine All Features

Now we'll merge all our feature sets into a single training dataset ready for machine learning.

**Final Dataset Will Include:**
- ✅ Base match information and targets
- ✅ Team form features (last 5 & 10 games)
- ✅ Team statistical features
- ✅ Head-to-head features
- ✅ League and context features

This comprehensive feature set will give our ML models the best chance to make accurate predictions.
"""

Base matches: 22,774 records (1 per match)
Team form: 45,548 records (2 per match - home & away team)
Team stats: ~84,000 records (2 per match)
H2H features: 22,774 records (1 per match) ✅
Context features: 22,774 records (1 per match) ✅

The issue is that team_form and team_stats have 2 records per match, but we're trying to merge them as if they have 1 record per match.

In [30]:
# Debug the duplicate fixture_id issue
print("🔍 DEBUGGING DUPLICATE FIXTURE_ID ISSUE")
print("=" * 50)

print("Base matches shape:", base_matches.shape)
print("Base matches fixture_id duplicates:", base_matches['fixture_id'].duplicated().sum())

print("\nH2H features shape:", h2h_features.shape)
print("H2H features fixture_id duplicates:", h2h_features['fixture_id'].duplicated().sum())

print("\nContext features shape:", context_features.shape)
print("Context features fixture_id duplicates:", context_features['fixture_id'].duplicated().sum())

print("\nContext features columns:", list(context_features.columns))

# Check if the issue is with the merged DataFrame after H2H
temp_df = base_matches.merge(h2h_features, on='fixture_id', how='left')
print("\nAfter H2H merge shape:", temp_df.shape)
print("After H2H merge fixture_id duplicates:", temp_df['fixture_id'].duplicated().sum())

# Check context_clean specifically
context_cols = [col for col in context_features.columns if col not in ['home_team_id', 'away_team_id', 'league_id', 'season_id', 'match_date']]
print("\nContext cols to keep:", context_cols)

context_clean = context_features[['fixture_id'] + context_cols].drop_duplicates(subset=['fixture_id'])
print("Context clean shape:", context_clean.shape)
print("Context clean fixture_id duplicates:", context_clean['fixture_id'].duplicated().sum())

🔍 DEBUGGING DUPLICATE FIXTURE_ID ISSUE
Base matches shape: (22774, 22)
Base matches fixture_id duplicates: 0

H2H features shape: (22774, 17)
H2H features fixture_id duplicates: 0

Context features shape: (22774, 19)
Context features fixture_id duplicates: 0

Context features columns: ['fixture_id', 'league_id', 'season_id', 'match_date', 'home_team_id', 'away_team_id', 'league_avg_total_goals', 'league_avg_over_2_5', 'league_avg_btts', 'league_avg_home_win', 'league_avg_draw', 'league_avg_away_win', 'season_avg_total_goals', 'season_avg_home_win', 'day_of_week', 'month', 'is_weekend', 'season_month', 'season_progress']

After H2H merge shape: (22774, 38)
After H2H merge fixture_id duplicates: 0

Context cols to keep: ['fixture_id', 'league_avg_total_goals', 'league_avg_over_2_5', 'league_avg_btts', 'league_avg_home_win', 'league_avg_draw', 'league_avg_away_win', 'season_avg_total_goals', 'season_avg_home_win', 'day_of_week', 'month', 'is_weekend', 'season_month', 'season_progress']
Co

In [31]:
# Check for duplicate column names in the merge
print("🔍 CHECKING FOR DUPLICATE COLUMN NAMES")
print("=" * 50)

# After H2H merge
temp_df = base_matches.merge(h2h_features, on='fixture_id', how='left')
print("Temp_df columns:", len(temp_df.columns))
print("Unique column names in temp_df:", len(set(temp_df.columns)))
print("Duplicate columns in temp_df:", [col for col in temp_df.columns if list(temp_df.columns).count(col) > 1])

# Context clean columns
context_cols = [col for col in context_features.columns if col not in ['home_team_id', 'away_team_id', 'league_id', 'season_id', 'match_date']]
context_clean = context_features[['fixture_id'] + context_cols].drop_duplicates(subset=['fixture_id'])
print("\nContext_clean columns:", len(context_clean.columns))
print("Unique column names in context_clean:", len(set(context_clean.columns)))
print("Duplicate columns in context_clean:", [col for col in context_clean.columns if list(context_clean.columns).count(col) > 1])

# Check for overlapping columns (besides fixture_id)
overlapping_cols = set(temp_df.columns) & set(context_clean.columns)
print("\nOverlapping columns between temp_df and context_clean:")
print(overlapping_cols)

🔍 CHECKING FOR DUPLICATE COLUMN NAMES
Temp_df columns: 38
Unique column names in temp_df: 38
Duplicate columns in temp_df: []

Context_clean columns: 15
Unique column names in context_clean: 14
Duplicate columns in context_clean: ['fixture_id', 'fixture_id']

Overlapping columns between temp_df and context_clean:
{'fixture_id'}


In [33]:
# Check what columns exist after each merge step
print("🔍 CHECKING COLUMN NAMES AFTER EACH MERGE")
print("=" * 50)

print("Base matches columns:")
print([col for col in base_matches.columns if 'team_id' in col])

print("\nH2H features columns:")
print([col for col in h2h_features.columns if 'team_id' in col])

# Test the merge step by step
temp_df = base_matches.merge(h2h_features, on='fixture_id', how='left')
print("\nAfter H2H merge columns:")
print([col for col in temp_df.columns if 'team_id' in col])
print("All columns:", list(temp_df.columns))

🔍 CHECKING COLUMN NAMES AFTER EACH MERGE
Base matches columns:
['home_team_id', 'away_team_id']

H2H features columns:
['home_team_id', 'away_team_id']

After H2H merge columns:
['home_team_id_x', 'away_team_id_x', 'home_team_id_y', 'away_team_id_y']
All columns: ['fixture_id', 'league_id', 'season_id', 'home_team_id_x', 'away_team_id_x', 'match_date', 'score_home', 'score_away', 'league_name', 'country_id', 'country_name', 'home_team_name', 'away_team_name', 'season_name', 'total_goals', 'goal_difference', 'result', 'over_2_5', 'btts', 'home_win', 'draw', 'away_win', 'home_team_id_y', 'away_team_id_y', 'h2h_games_played', 'h2h_home_wins', 'h2h_draws', 'h2h_away_wins', 'h2h_recent_games', 'h2h_recent_home_wins', 'h2h_recent_draws', 'h2h_recent_away_wins', 'h2h_home_win_rate', 'h2h_draw_rate', 'h2h_away_win_rate', 'h2h_avg_home_goals', 'h2h_avg_away_goals', 'h2h_avg_total_goals']


the H2H merge created duplicate columns with _x and _y suffixes. 
The team_id columns are now named home_team_id_x and away_team_id_x instead of home_team_id and away_team_id



The key fix is that I now drop the duplicate home_team_id and away_team_id columns from h2h_features before merging, so they don't conflict with the ones in base_matches.


The issue was that both DataFrames had home_team_id and away_team_id columns, causing pandas to create _x and _y suffixes, which then broke the subsequent references to these columns.

In [35]:
# CELL 14 - CODE: Combine All Features into Final Dataset
def create_final_dataset(base_df, team_form, team_stats, h2h_features, context_features):
    """Combine all features into final training dataset"""
    
    print("🔗 COMBINING ALL FEATURES")
    print("=" * 40)
    
    # Start with base matches
    final_df = base_df.copy()
    
    # Add head-to-head features (drop duplicate columns to avoid _x, _y suffixes)
    h2h_clean = h2h_features.drop(['home_team_id', 'away_team_id'], axis=1)
    final_df = final_df.merge(h2h_clean, on='fixture_id', how='left')
    print(f"✅ Added H2H features: {len([col for col in h2h_clean.columns if col.startswith('h2h_')])}")
    
    # Add context features (deduplicate first and fix column selection)
    context_cols = [col for col in context_features.columns if col not in ['home_team_id', 'away_team_id', 'league_id', 'season_id', 'match_date', 'fixture_id']]
    context_clean = context_features[['fixture_id'] + context_cols].drop_duplicates(subset=['fixture_id'])
    final_df = final_df.merge(context_clean, on='fixture_id', how='left')
    print(f"✅ Added context features: {len(context_cols)}")
    
    # Add team form features for home team
    home_form = team_form[team_form['is_home'] == True].copy()
    home_form_cols = [col for col in home_form.columns if col.startswith('form_')]
    home_form_renamed = home_form[['fixture_id'] + home_form_cols].copy()
    home_form_renamed.columns = ['fixture_id'] + [f'home_{col}' for col in home_form_cols]
    final_df = final_df.merge(home_form_renamed, on='fixture_id', how='left')
    print(f"✅ Added home team form features: {len(home_form_cols)}")
    
    # Add team form features for away team
    away_form = team_form[team_form['is_home'] == False].copy()
    away_form_cols = [col for col in away_form.columns if col.startswith('form_')]
    away_form_renamed = away_form[['fixture_id'] + away_form_cols].copy()
    away_form_renamed.columns = ['fixture_id'] + [f'away_{col}' for col in away_form_cols]
    final_df = final_df.merge(away_form_renamed, on='fixture_id', how='left')
    print(f"✅ Added away team form features: {len(away_form_cols)}")
    
    # Add team statistics for home team
    home_stats = team_stats.merge(
        final_df[['fixture_id', 'home_team_id']], 
        left_on=['fixture_id', 'team_id'], 
        right_on=['fixture_id', 'home_team_id'],
        how='inner'
    )
    stat_cols = [col for col in team_stats.columns if col.startswith('stat_')]
    home_stats_renamed = home_stats[['fixture_id'] + stat_cols].copy()
    home_stats_renamed.columns = ['fixture_id'] + [f'home_{col}' for col in stat_cols]
    final_df = final_df.merge(home_stats_renamed, on='fixture_id', how='left')
    print(f"✅ Added home team statistics: {len(stat_cols)}")
    
    # Add team statistics for away team  
    away_stats = team_stats.merge(
        final_df[['fixture_id', 'away_team_id']], 
        left_on=['fixture_id', 'team_id'], 
        right_on=['fixture_id', 'away_team_id'],
        how='inner'
    )
    away_stats_renamed = away_stats[['fixture_id'] + stat_cols].copy()
    away_stats_renamed.columns = ['fixture_id'] + [f'away_{col}' for col in stat_cols]
    final_df = final_df.merge(away_stats_renamed, on='fixture_id', how='left')
    print(f"✅ Added away team statistics: {len(stat_cols)}")
    
    # Fill missing values
    feature_cols = [col for col in final_df.columns if any(col.startswith(prefix) for prefix in ['home_form_', 'away_form_', 'home_stat_', 'away_stat_', 'h2h_', 'league_', 'season_'])]
    final_df[feature_cols] = final_df[feature_cols].fillna(0)
    
    # Create additional derived features
    final_df['form_5_goal_diff_advantage'] = final_df['home_form_5_goal_diff_avg'] - final_df['away_form_5_goal_diff_avg']
    final_df['form_10_goal_diff_advantage'] = final_df['home_form_10_goal_diff_avg'] - final_df['away_form_10_goal_diff_avg']
    final_df['form_5_points_advantage'] = final_df['home_form_5_points_per_game'] - final_df['away_form_5_points_per_game']
    final_df['form_10_points_advantage'] = final_df['home_form_10_points_per_game'] - final_df['away_form_10_points_per_game']
    
    # Remove rows with insufficient data for reliable predictions
    final_df = final_df[
        (final_df['home_form_5_games_played'] >= 3) & 
        (final_df['away_form_5_games_played'] >= 3)
    ].copy()
    
    print(f"\n📊 FINAL DATASET SUMMARY")
    print("=" * 40)
    print(f"Total matches: {len(final_df):,}")
    print(f"Total features: {len(feature_cols) + 4}")  # +4 for derived features
    print(f"Date range: {final_df['match_date'].min()} to {final_df['match_date'].max()}")
    
    # Feature categories
    form_features = len([col for col in final_df.columns if 'form_' in col])
    stat_features = len([col for col in final_df.columns if 'stat_' in col])
    h2h_features = len([col for col in final_df.columns if 'h2h_' in col])
    context_features = len([col for col in final_df.columns if any(x in col for x in ['league_', 'season_', 'day_', 'month', 'weekend', 'progress'])])
    
    print(f"\nFeature breakdown:")
    print(f"  Form features: {form_features}")
    print(f"  Statistical features: {stat_features}")
    print(f"  Head-to-head features: {h2h_features}")
    print(f"  Context features: {context_features}")
    print(f"  Derived features: 4")
    
    return final_df

# Create the final dataset
print("🚀 Creating final combined dataset...")
final_dataset = create_final_dataset(base_matches, team_form, team_stats, h2h_features, context_features)


🚀 Creating final combined dataset...
🔗 COMBINING ALL FEATURES
✅ Added H2H features: 14
✅ Added context features: 13
✅ Added home team form features: 20
✅ Added away team form features: 20
✅ Added home team statistics: 9
✅ Added away team statistics: 9

📊 FINAL DATASET SUMMARY
Total matches: 20,309
Total features: 90
Date range: 2022-06-11 19:00:00 to 2025-05-29 16:00:00

Feature breakdown:
  Form features: 44
  Statistical features: 18
  Head-to-head features: 14
  Context features: 17
  Derived features: 4


## 🎯 Step 2.7: Data Quality Check & Train/Test Split

Before we move to modeling, let's ensure our data quality is good and create proper train/test splits for evaluation.

**Final Checks:**
- Feature completeness and distributions
- Target variable balance
- Temporal train/test split (crucial for time-series data)
- Feature correlation analysis
"""

In [37]:
# CELL 16 - CODE: Data Quality Check and Train/Test Split
def perform_data_quality_check(final_df):
    """Perform final data quality checks and create train/test split"""
    
    print("🎯 FINAL DATA QUALITY CHECK")
    print("=" * 50)
    
    # Check for missing values
    missing_data = final_df.isnull().sum()
    missing_features = missing_data[missing_data > 0]
    
    if len(missing_features) > 0:
        print("⚠️ Features with missing values:")
        for feature, count in missing_features.items():
            pct = count / len(final_df) * 100
            print(f"   {feature}: {count} ({pct:.1f}%)")
    else:
        print("✅ No missing values in feature set!")
    
    # Target distribution
    print(f"\n🎯 Target Variable Distribution:")
    print(f"   Home wins: {final_df['home_win'].mean():.3f}")
    print(f"   Draws: {final_df['draw'].mean():.3f}")
    print(f"   Away wins: {final_df['away_win'].mean():.3f}")
    print(f"   Over 2.5 goals: {final_df['over_2_5'].mean():.3f}")
    print(f"   Both teams score: {final_df['btts'].mean():.3f}")
    
    # Feature statistics - only numeric features
    feature_cols = [col for col in final_df.columns if any(col.startswith(prefix) for prefix in 
                   ['home_form_', 'away_form_', 'home_stat_', 'away_stat_', 'h2h_', 'league_avg_', 'season_avg_', 'day_', 'month', 'weekend', 'progress', 'form_'])]
    
    # Remove any non-numeric columns that might have slipped through
    numeric_feature_cols = []
    for col in feature_cols:
        try:
            pd.to_numeric(final_df[col], errors='raise')
            numeric_feature_cols.append(col)
        except:
            print(f"   Skipping non-numeric column: {col}")
    
    feature_cols = numeric_feature_cols
    
    print(f"\n📊 Feature Statistics:")
    print(f"   Total features: {len(feature_cols)}")
    print(f"   Features with variance > 0: {(final_df[feature_cols].var() > 0).sum()}")
    print(f"   Features with all zeros: {(final_df[feature_cols].sum() == 0).sum()}")
    
    # Create temporal train/test split
    final_df_sorted = final_df.sort_values('match_date')
    
    # Use 80% for training, 20% for testing (temporal split)
    split_date = final_df_sorted['match_date'].quantile(0.8)
    
    train_df = final_df_sorted[final_df_sorted['match_date'] <= split_date].copy()
    test_df = final_df_sorted[final_df_sorted['match_date'] > split_date].copy()
    
    print(f"\n📅 Train/Test Split:")
    print(f"   Split date: {split_date}")
    print(f"   Training set: {len(train_df):,} matches ({len(train_df)/len(final_df):.1%})")
    print(f"   Test set: {len(test_df):,} matches ({len(test_df)/len(final_df):.1%})")
    print(f"   Training date range: {train_df['match_date'].min()} to {train_df['match_date'].max()}")
    print(f"   Test date range: {test_df['match_date'].min()} to {test_df['match_date'].max()}")
    
    # Prepare feature matrices
    X_train = train_df[feature_cols].values
    y_train_multiclass = train_df['result'].values  # H, D, A
    y_train_home = train_df['home_win'].values
    y_train_draw = train_df['draw'].values
    y_train_away = train_df['away_win'].values
    y_train_over25 = train_df['over_2_5'].values
    y_train_btts = train_df['btts'].values
    
    X_test = test_df[feature_cols].values
    y_test_multiclass = test_df['result'].values
    y_test_home = test_df['home_win'].values
    y_test_draw = test_df['draw'].values
    y_test_away = test_df['away_win'].values
    y_test_over25 = test_df['over_2_5'].values
    y_test_btts = test_df['btts'].values
    
    print(f"\n🧮 Feature Matrix Shapes:")
    print(f"   X_train: {X_train.shape}")
    print(f"   X_test: {X_test.shape}")
    print(f"   Feature names: {len(feature_cols)} features")
    
    return {
        'train_df': train_df,
        'test_df': test_df,
        'X_train': X_train,
        'X_test': X_test,
        'y_train_multiclass': y_train_multiclass,
        'y_test_multiclass': y_test_multiclass,
        'y_train_home': y_train_home,
        'y_test_home': y_test_home,
        'y_train_draw': y_train_draw,
        'y_test_draw': y_test_draw,
        'y_train_away': y_train_away,
        'y_test_away': y_test_away,
        'y_train_over25': y_train_over25,
        'y_test_over25': y_test_over25,
        'y_train_btts': y_train_btts,
        'y_test_btts': y_test_btts,
        'feature_names': feature_cols,
        'split_date': split_date
    }

# Perform quality check and create splits
ml_data = perform_data_quality_check(final_dataset)


🎯 FINAL DATA QUALITY CHECK
✅ No missing values in feature set!

🎯 Target Variable Distribution:
   Home wins: 0.442
   Draws: 0.252
   Away wins: 0.306
   Over 2.5 goals: 0.523
   Both teams score: 0.530

📊 Feature Statistics:
   Total features: 86
   Features with variance > 0: 86
   Features with all zeros: 0

📅 Train/Test Split:
   Split date: 2024-11-10 19:45:00
   Training set: 16,248 matches (80.0%)
   Test set: 4,061 matches (20.0%)
   Training date range: 2022-06-11 19:00:00 to 2024-11-10 19:45:00
   Test date range: 2024-11-10 20:00:00 to 2025-05-29 16:00:00

🧮 Feature Matrix Shapes:
   X_train: (16248, 86)
   X_test: (4061, 86)
   Feature names: 86 features


## 💾 Step 2.8: Save Prepared Data

Let's save our prepared dataset and create a summary of what we've accomplished.

**What We've Built:**
- ✅ Comprehensive feature engineering from 155k+ matches
- ✅ Team form features (recent performance indicators)
- ✅ Statistical features (playing style and strength)
- ✅ Head-to-head historical records
- ✅ League and contextual features
- ✅ Proper temporal train/test split
- ✅ Multiple target variables for different betting markets

**Ready for:** XGBoost, SVM, Random Forest, Neural Networks in Step 3!
"""


In [38]:
# CELL 18 - CODE: Save Data and Create Summary
import pickle
from datetime import datetime

def save_prepared_data(ml_data, final_dataset):
    """Save the prepared data and create summary"""
    
    print("💾 SAVING PREPARED DATA")
    print("=" * 40)
    
    # Save the data dictionary
    with open('ml_data_prepared.pkl', 'wb') as f:
        pickle.dump(ml_data, f)
    
    # Save the full dataset as CSV for inspection
    final_dataset.to_csv('final_dataset.csv', index=False)
    
    # Create feature summary
    feature_summary = {
        'total_features': len(ml_data['feature_names']),
        'training_samples': ml_data['X_train'].shape[0],
        'test_samples': ml_data['X_test'].shape[0],
        'split_date': ml_data['split_date'],
        'target_distributions': {
            'home_win_rate': ml_data['y_train_home'].mean(),
            'draw_rate': ml_data['y_train_draw'].mean(),
            'away_win_rate': ml_data['y_train_away'].mean(),
            'over_2_5_rate': ml_data['y_train_over25'].mean(),
            'btts_rate': ml_data['y_train_btts'].mean()
        },
        'feature_categories': {
            'form_features': len([f for f in ml_data['feature_names'] if 'form_' in f]),
            'statistical_features': len([f for f in ml_data['feature_names'] if 'stat_' in f]),
            'h2h_features': len([f for f in ml_data['feature_names'] if 'h2h_' in f]),
            'context_features': len([f for f in ml_data['feature_names'] if any(x in f for x in ['league_', 'season_', 'day_', 'month', 'weekend', 'progress'])])
        },
        'created_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    
    with open('feature_summary.pkl', 'wb') as f:
        pickle.dump(feature_summary, f)
    
    print("✅ Data saved successfully!")
    print(f"   ml_data_prepared.pkl: ML-ready data dictionary")
    print(f"   final_dataset.csv: Full dataset for inspection")
    print(f"   feature_summary.pkl: Feature engineering summary")
    
    return feature_summary

# Save everything
feature_summary = save_prepared_data(ml_data, final_dataset)

print(f"\n🎉 STEP 2 COMPLETE!")
print("=" * 50)
print(f"✅ Feature engineering successful!")
print(f"✅ {feature_summary['total_features']} features created")
print(f"✅ {feature_summary['training_samples']:,} training samples")
print(f"✅ {feature_summary['test_samples']:,} test samples")
print(f"✅ Data saved and ready for modeling")

print(f"\n🚀 Ready for Step 3: Model Training!")
print("   Next: XGBoost → SVM → Random Forest → Neural Networks")

# Show top features for preview
print(f"\n📋 Feature Preview (first 10):")
for i, feature in enumerate(ml_data['feature_names'][:10]):
    print(f"   {i+1:2d}. {feature}")
print(f"   ... and {len(ml_data['feature_names'])-10} more features")

💾 SAVING PREPARED DATA
✅ Data saved successfully!
   ml_data_prepared.pkl: ML-ready data dictionary
   final_dataset.csv: Full dataset for inspection
   feature_summary.pkl: Feature engineering summary

🎉 STEP 2 COMPLETE!
✅ Feature engineering successful!
✅ 86 features created
✅ 16,248 training samples
✅ 4,061 test samples
✅ Data saved and ready for modeling

🚀 Ready for Step 3: Model Training!
   Next: XGBoost → SVM → Random Forest → Neural Networks

📋 Feature Preview (first 10):
    1. h2h_games_played
    2. h2h_home_wins
    3. h2h_draws
    4. h2h_away_wins
    5. h2h_recent_games
    6. h2h_recent_home_wins
    7. h2h_recent_draws
    8. h2h_recent_away_wins
    9. h2h_home_win_rate
   10. h2h_draw_rate
   ... and 76 more features


# 🤖 Step 3: Machine Learning Model Training

**Goal**: Train and evaluate 4 different ML models for football betting predictions

**Models to implement:**
1. **XGBoost** - Gradient boosting (excellent for tabular data)
2. **SVM** - Support Vector Machine (robust classifier)
3. **Random Forest** - Ensemble method (good baseline)
4. **Neural Networks** - Deep learning (complex patterns)

**Evaluation metrics:**
- **Accuracy** - Overall prediction correctness
- **Precision/Recall** - Per-class performance
- **F1-Score** - Balanced metric
- **Log Loss** - Probability calibration
- **ROI simulation** - Betting profitability

From Step 2, we have:
- ✅ 86 engineered features
- ✅ 16,248 training samples
- ✅ 4,061 test samples
- ✅ Balanced target distributions
"""

again, if we wanted a more clean setup, we could create a new file called training, where we would set up the enviroment like this

In [39]:
# CELL 2 - CODE: Setup and Load Data
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, log_loss
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import xgboost as xgb
from sklearn.neural_network import MLPClassifier
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('default')
sns.set_palette("husl")

print("🤖 MACHINE LEARNING MODEL TRAINING")
print("=" * 50)

# Load prepared data
try:
    with open('ml_data_prepared.pkl', 'rb') as f:
        ml_data = pickle.load(f)
    
    print("✅ Data loaded successfully!")
    print(f"📊 Training samples: {ml_data['X_train'].shape[0]:,}")
    print(f"📊 Test samples: {ml_data['X_test'].shape[0]:,}")
    print(f"📊 Features: {ml_data['X_train'].shape[1]}")
    print(f"📊 Split date: {ml_data['split_date']}")
    
except FileNotFoundError:
    print("❌ ml_data_prepared.pkl not found. Please run Step 2 first.")


🤖 MACHINE LEARNING MODEL TRAINING
✅ Data loaded successfully!
📊 Training samples: 16,248
📊 Test samples: 4,061
📊 Features: 86
📊 Split date: 2024-11-10 19:45:00


## 🎯 Step 3.1: Baseline Analysis

Before training complex models, let's establish baselines and understand our data distribution.

**Baseline metrics:**
- Random prediction accuracy
- Most frequent class accuracy
- Feature importance preview
"""

In [40]:
# CELL 4 - CODE: Baseline Analysis and Data Preparation
def analyze_baselines_and_prepare_data(ml_data):
    """Analyze baseline performance and prepare data for modeling"""
    
    print("🎯 BASELINE ANALYSIS")
    print("=" * 40)
    
    # Extract data
    X_train = ml_data['X_train']
    X_test = ml_data['X_test']
    y_train = ml_data['y_train_multiclass']
    y_test = ml_data['y_test_multiclass']
    
    # Baseline accuracies
    unique_classes, class_counts = np.unique(y_train, return_counts=True)
    most_frequent_class = unique_classes[np.argmax(class_counts)]
    baseline_accuracy = np.max(class_counts) / len(y_train)
    random_accuracy = 1 / len(unique_classes)
    
    print(f"📊 Class distribution in training:")
    for cls, count in zip(unique_classes, class_counts):
        print(f"   {cls}: {count:,} ({count/len(y_train):.3f})")
    
    print(f"\n🎲 Baseline accuracies:")
    print(f"   Random prediction: {random_accuracy:.3f}")
    print(f"   Most frequent class ({most_frequent_class}): {baseline_accuracy:.3f}")
    
    # Encode labels for sklearn
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    y_test_encoded = le.transform(y_test)
    
    print(f"\n🔢 Label encoding:")
    for i, cls in enumerate(le.classes_):
        print(f"   {cls} → {i}")
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    print(f"\n📏 Feature scaling completed:")
    print(f"   Training mean: {X_train_scaled.mean():.3f}")
    print(f"   Training std: {X_train_scaled.std():.3f}")
    
    return {
        'X_train': X_train,
        'X_test': X_test,
        'X_train_scaled': X_train_scaled,
        'X_test_scaled': X_test_scaled,
        'y_train': y_train_encoded,
        'y_test': y_test_encoded,
        'label_encoder': le,
        'scaler': scaler,
        'baseline_accuracy': baseline_accuracy,
        'random_accuracy': random_accuracy,
        'feature_names': ml_data['feature_names']
    }

# Prepare data for modeling
model_data = analyze_baselines_and_prepare_data(ml_data)


🎯 BASELINE ANALYSIS
📊 Class distribution in training:
   A: 4,977 (0.306)
   D: 4,074 (0.251)
   H: 7,197 (0.443)

🎲 Baseline accuracies:
   Random prediction: 0.333
   Most frequent class (H): 0.443

🔢 Label encoding:
   A → 0
   D → 1
   H → 2

📏 Feature scaling completed:
   Training mean: 0.000
   Training std: 1.000


## 🌳 Step 3.2: XGBoost Model

Starting with XGBoost - often the best performer for tabular data like ours.

**XGBoost advantages:**
- Excellent for structured/tabular data
- Built-in feature importance
- Handles missing values well
- Fast training and prediction
"""

In [41]:
# CELL 6 - CODE: XGBoost Model Training and Evaluation
def train_xgboost_model(model_data):
    """Train and evaluate XGBoost model"""
    
    print("🌳 XGBOOST MODEL TRAINING")
    print("=" * 40)
    
    # XGBoost works well with unscaled data
    X_train = model_data['X_train']
    X_test = model_data['X_test']
    y_train = model_data['y_train']
    y_test = model_data['y_test']
    
    # Create XGBoost classifier
    xgb_model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='mlogloss',
        use_label_encoder=False
    )
    
    print("🔄 Training XGBoost model...")
    
    # Train model
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )
    
    # Predictions
    y_pred_train = xgb_model.predict(X_train)
    y_pred_test = xgb_model.predict(X_test)
    y_prob_test = xgb_model.predict_proba(X_test)
    
    # Calculate metrics
    train_accuracy = (y_pred_train == y_train).mean()
    test_accuracy = (y_pred_test == y_test).mean()
    
    # Cross-validation score
    cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=5, scoring='accuracy')
    
    print(f"✅ XGBoost training completed!")
    print(f"📊 Training accuracy: {train_accuracy:.4f}")
    print(f"📊 Test accuracy: {test_accuracy:.4f}")
    print(f"📊 CV accuracy: {cv_scores.mean():.4f} (±{cv_scores.std()*2:.4f})")
    print(f"📊 Baseline beat by: {test_accuracy - model_data['baseline_accuracy']:.4f}")
    
    # Feature importance
    feature_importance = pd.DataFrame({
        'feature': model_data['feature_names'],
        'importance': xgb_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(f"\n🎯 Top 10 most important features:")
    for i, (_, row) in enumerate(feature_importance.head(10).iterrows(), 1):
        print(f"   {i:2d}. {row['feature']}: {row['importance']:.4f}")
    
    # Classification report
    le = model_data['label_encoder']
    print(f"\n📋 Detailed classification report:")
    print(classification_report(y_test, y_pred_test, target_names=le.classes_))
    
    return {
        'model': xgb_model,
        'train_accuracy': train_accuracy,
        'test_accuracy': test_accuracy,
        'cv_scores': cv_scores,
        'predictions': y_pred_test,
        'probabilities': y_prob_test,
        'feature_importance': feature_importance
    }

# Train XGBoost model
xgb_results = train_xgboost_model(model_data)


🌳 XGBOOST MODEL TRAINING
🔄 Training XGBoost model...
✅ XGBoost training completed!
📊 Training accuracy: 0.8922
📊 Test accuracy: 0.6397
📊 CV accuracy: 0.6376 (±0.0247)
📊 Baseline beat by: 0.1968

🎯 Top 10 most important features:
    1. home_stat_goals: 0.0860
    2. away_stat_goals: 0.0833
    3. away_stat_assists: 0.0597
    4. home_stat_assists: 0.0579
    5. form_10_goal_diff_advantage: 0.0198
    6. home_stat_redcards: 0.0150
    7. away_stat_redcards: 0.0149
    8. league_avg_draw: 0.0114
    9. league_avg_away_win: 0.0112
   10. form_10_points_advantage: 0.0109

📋 Detailed classification report:
              precision    recall  f1-score   support

           A       0.68      0.67      0.68      1231
           D       0.45      0.34      0.39      1052
           H       0.69      0.80      0.74      1778

    accuracy                           0.64      4061
   macro avg       0.61      0.60      0.60      4061
weighted avg       0.62      0.64      0.63      4061



when looking at the prediction scores here, the xgboost actually performs really well, only struggling with the prediction of a draw, whihc is common/a typical issue!

## 🎯 Step 3.3: Support Vector Machine (SVM)

SVM with RBF kernel for non-linear pattern recognition. this is just a test compared to previous literature

**SVM advantages:**
- Effective in high-dimensional spaces
- Memory efficient
- Versatile (different kernels)
- Good for complex decision boundaries
"""

In [42]:
# CELL 8 - CODE: SVM Model Training and Evaluation
def train_svm_model(model_data):
    """Train and evaluate SVM model"""
    
    print("🎯 SVM MODEL TRAINING")
    print("=" * 40)
    
    # SVM requires scaled features
    X_train = model_data['X_train_scaled']
    X_test = model_data['X_test_scaled']
    y_train = model_data['y_train']
    y_test = model_data['y_test']
    
    # Create SVM classifier with probability estimates
    svm_model = SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        probability=True,  # Enable probability estimates
        random_state=42
    )
    
    print("🔄 Training SVM model...")
    
    # Train model
    svm_model.fit(X_train, y_train)
    
    # Predictions
    y_pred_train = svm_model.predict(X_train)
    y_pred_test = svm_model.predict(X_test)
    y_prob_test = svm_model.predict_proba(X_test)
    
    # Calculate metrics
    train_accuracy = (y_pred_train == y_train).mean()
    test_accuracy = (y_pred_test == y_test).mean()
    
    # Cross-validation score (smaller sample for speed)
    cv_scores = cross_val_score(svm_model, X_train[:5000], y_train[:5000], cv=3, scoring='accuracy')
    
    print(f"✅ SVM training completed!")
    print(f"📊 Training accuracy: {train_accuracy:.4f}")
    print(f"📊 Test accuracy: {test_accuracy:.4f}")
    print(f"📊 CV accuracy (sample): {cv_scores.mean():.4f} (±{cv_scores.std()*2:.4f})")
    print(f"📊 Baseline beat by: {test_accuracy - model_data['baseline_accuracy']:.4f}")
    
    # Classification report
    le = model_data['label_encoder']
    print(f"\n📋 Detailed classification report:")
    print(classification_report(y_test, y_pred_test, target_names=le.classes_))
    
    return {
        'model': svm_model,
        'train_accuracy': train_accuracy,
        'test_accuracy': test_accuracy,
        'cv_scores': cv_scores,
        'predictions': y_pred_test,
        'probabilities': y_prob_test
    }

# Train SVM model
svm_results = train_svm_model(model_data)


🎯 SVM MODEL TRAINING
🔄 Training SVM model...
✅ SVM training completed!
📊 Training accuracy: 0.7085
📊 Test accuracy: 0.5836
📊 CV accuracy (sample): 0.5910 (±0.0067)
📊 Baseline beat by: 0.1407

📋 Detailed classification report:
              precision    recall  f1-score   support

           A       0.59      0.64      0.61      1231
           D       0.38      0.19      0.25      1052
           H       0.63      0.78      0.69      1778

    accuracy                           0.58      4061
   macro avg       0.53      0.54      0.52      4061
weighted avg       0.55      0.58      0.56      4061



## 🌲 Step 3.4: Random Forest

Random Forest as our ensemble baseline - reliable and interpretable.

**Random Forest advantages:**
- Robust to overfitting
- Feature importance available
- Handles mixed data types well
- Good baseline performance
"""

In [43]:
# CELL 10 - CODE: Random Forest Model Training and Evaluation
def train_random_forest_model(model_data):
    """Train and evaluate Random Forest model"""
    
    print("🌲 RANDOM FOREST MODEL TRAINING")
    print("=" * 40)
    
    # Random Forest works well with unscaled data
    X_train = model_data['X_train']
    X_test = model_data['X_test']
    y_train = model_data['y_train']
    y_test = model_data['y_test']
    
    # Create Random Forest classifier
    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    )
    
    print("🔄 Training Random Forest model...")
    
    # Train model
    rf_model.fit(X_train, y_train)
    
    # Predictions
    y_pred_train = rf_model.predict(X_train)
    y_pred_test = rf_model.predict(X_test)
    y_prob_test = rf_model.predict_proba(X_test)
    
    # Calculate metrics
    train_accuracy = (y_pred_train == y_train).mean()
    test_accuracy = (y_pred_test == y_test).mean()
    
    # Cross-validation score
    cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='accuracy')
    
    print(f"✅ Random Forest training completed!")
    print(f"📊 Training accuracy: {train_accuracy:.4f}")
    print(f"📊 Test accuracy: {test_accuracy:.4f}")
    print(f"📊 CV accuracy: {cv_scores.mean():.4f} (±{cv_scores.std()*2:.4f})")
    print(f"📊 Baseline beat by: {test_accuracy - model_data['baseline_accuracy']:.4f}")
    
    # Feature importance
    feature_importance = pd.DataFrame({
        'feature': model_data['feature_names'],
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(f"\n🎯 Top 10 most important features:")
    for i, (_, row) in enumerate(feature_importance.head(10).iterrows(), 1):
        print(f"   {i:2d}. {row['feature']}: {row['importance']:.4f}")
    
    # Classification report
    le = model_data['label_encoder']
    print(f"\n📋 Detailed classification report:")
    print(classification_report(y_test, y_pred_test, target_names=le.classes_))
    
    return {
        'model': rf_model,
        'train_accuracy': train_accuracy,
        'test_accuracy': test_accuracy,
        'cv_scores': cv_scores,
        'predictions': y_pred_test,
        'probabilities': y_prob_test,
        'feature_importance': feature_importance
    }

# Train Random Forest model
rf_results = train_random_forest_model(model_data)


🌲 RANDOM FOREST MODEL TRAINING
🔄 Training Random Forest model...
✅ Random Forest training completed!
📊 Training accuracy: 0.8508
📊 Test accuracy: 0.6055
📊 CV accuracy: 0.6162 (±0.0189)
📊 Baseline beat by: 0.1626

🎯 Top 10 most important features:
    1. away_stat_goals: 0.1193
    2. home_stat_goals: 0.0974
    3. away_stat_assists: 0.0707
    4. home_stat_assists: 0.0624
    5. form_10_goal_diff_advantage: 0.0308
    6. form_10_points_advantage: 0.0227
    7. home_form_10_goal_diff_avg: 0.0185
    8. form_5_goal_diff_advantage: 0.0176
    9. season_avg_home_win: 0.0156
   10. away_form_10_goals_for_avg: 0.0150

📋 Detailed classification report:
              precision    recall  f1-score   support

           A       0.64      0.63      0.63      1231
           D       0.42      0.15      0.23      1052
           H       0.62      0.86      0.72      1778

    accuracy                           0.61      4061
   macro avg       0.56      0.55      0.53      4061
weighted avg       0

## 🧠 Step 3.5: Neural Network

Multi-layer Perceptron for capturing complex non-linear patterns.

**Neural Network advantages:**
- Can learn complex patterns
- Non-linear decision boundaries
- Good for large datasets
- Flexible architecture
"""


In [44]:
def train_neural_network_model(model_data):
    """Train and evaluate Neural Network model"""
    
    print("🧠 NEURAL NETWORK MODEL TRAINING")
    print("=" * 40)
    
    # Neural networks require scaled features
    X_train = model_data['X_train_scaled']
    X_test = model_data['X_test_scaled']
    y_train = model_data['y_train']
    y_test = model_data['y_test']
    
    # Create Multi-layer Perceptron classifier
    nn_model = MLPClassifier(
        hidden_layer_sizes=(128, 64, 32),
        activation='relu',
        solver='adam',
        alpha=0.001,
        batch_size=128,
        learning_rate='adaptive',
        max_iter=500,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1
    )
    
    print("🔄 Training Neural Network model...")
    
    # Train model
    nn_model.fit(X_train, y_train)
    
    # Predictions
    y_pred_train = nn_model.predict(X_train)
    y_pred_test = nn_model.predict(X_test)
    y_prob_test = nn_model.predict_proba(X_test)
    
    # Calculate metrics
    train_accuracy = (y_pred_train == y_train).mean()
    test_accuracy = (y_pred_test == y_test).mean()
    
    # Cross-validation score (smaller sample for speed)
    cv_scores = cross_val_score(nn_model, X_train[:8000], y_train[:8000], cv=3, scoring='accuracy')
    
    print(f"✅ Neural Network training completed!")
    print(f"📊 Training accuracy: {train_accuracy:.4f}")
    print(f"📊 Test accuracy: {test_accuracy:.4f}")
    print(f"📊 CV accuracy (sample): {cv_scores.mean():.4f} (±{cv_scores.std()*2:.4f})")
    print(f"📊 Baseline beat by: {test_accuracy - model_data['baseline_accuracy']:.4f}")
    print(f"📊 Training iterations: {nn_model.n_iter_}")
    
    # Classification report
    le = model_data['label_encoder']
    print(f"\n📋 Detailed classification report:")
    print(classification_report(y_test, y_pred_test, target_names=le.classes_))
    
    return {
        'model': nn_model,
        'train_accuracy': train_accuracy,
        'test_accuracy': test_accuracy,
        'cv_scores': cv_scores,
        'predictions': y_pred_test,
        'probabilities': y_prob_test
    }

# Train Neural Network model
nn_results = train_neural_network_model(model_data)


🧠 NEURAL NETWORK MODEL TRAINING
🔄 Training Neural Network model...
✅ Neural Network training completed!
📊 Training accuracy: 0.7944
📊 Test accuracy: 0.5762
📊 CV accuracy (sample): 0.5913 (±0.0122)
📊 Baseline beat by: 0.1333
📊 Training iterations: 29

📋 Detailed classification report:
              precision    recall  f1-score   support

           A       0.62      0.59      0.60      1231
           D       0.37      0.36      0.36      1052
           H       0.66      0.70      0.68      1778

    accuracy                           0.58      4061
   macro avg       0.55      0.55      0.55      4061
weighted avg       0.57      0.58      0.57      4061



## 📊 Step 3.6: Model Comparison and Analysis

Now let's compare all four models and identify the best performer.

**Comparison metrics:**
- Test accuracy
- Cross-validation stability
- Prediction confidence
- Feature importance insights
"""


In [45]:
# CELL 14 - CODE: Model Comparison and Performance Analysis
def compare_all_models(xgb_results, svm_results, rf_results, nn_results, model_data):
    """Compare performance of all trained models"""
    
    print("📊 MODEL COMPARISON ANALYSIS")
    print("=" * 50)
    
    # Create comparison dataframe
    models_comparison = pd.DataFrame({
        'Model': ['XGBoost', 'SVM', 'Random Forest', 'Neural Network'],
        'Test_Accuracy': [
            xgb_results['test_accuracy'],
            svm_results['test_accuracy'],
            rf_results['test_accuracy'],
            nn_results['test_accuracy']
        ],
        'Train_Accuracy': [
            xgb_results['train_accuracy'],
            svm_results['train_accuracy'],
            rf_results['train_accuracy'],
            nn_results['train_accuracy']
        ],
        'CV_Mean': [
            xgb_results['cv_scores'].mean(),
            svm_results['cv_scores'].mean(),
            rf_results['cv_scores'].mean(),
            nn_results['cv_scores'].mean()
        ],
        'CV_Std': [
            xgb_results['cv_scores'].std(),
            svm_results['cv_scores'].std(),
            rf_results['cv_scores'].std(),
            nn_results['cv_scores'].std()
        ]
    })
    
    # Calculate overfitting (train - test accuracy)
    models_comparison['Overfitting'] = models_comparison['Train_Accuracy'] - models_comparison['Test_Accuracy']
    
    # Sort by test accuracy
    models_comparison = models_comparison.sort_values('Test_Accuracy', ascending=False)
    
    print("🏆 MODEL PERFORMANCE RANKING")
    print("-" * 30)
    
    for i, (_, row) in enumerate(models_comparison.iterrows(), 1):
        print(f"{i}. {row['Model']}:")
        print(f"   Test Accuracy: {row['Test_Accuracy']:.4f}")
        print(f"   CV Score: {row['CV_Mean']:.4f} (±{row['CV_Std']:.4f})")
        print(f"   Overfitting: {row['Overfitting']:.4f}")
        print()
    
    # Best model
    best_model_name = models_comparison.iloc[0]['Model']
    best_accuracy = models_comparison.iloc[0]['Test_Accuracy']
    baseline_accuracy = model_data['baseline_accuracy']
    
    print(f"🥇 BEST MODEL: {best_model_name}")
    print(f"   Accuracy: {best_accuracy:.4f}")
    print(f"   Improvement over baseline: {best_accuracy - baseline_accuracy:.4f}")
    print(f"   Improvement percentage: {((best_accuracy - baseline_accuracy) / baseline_accuracy * 100):.1f}%")
    
    # Feature importance comparison (for tree-based models)
    if 'feature_importance' in xgb_results and 'feature_importance' in rf_results:
        print(f"\n🎯 FEATURE IMPORTANCE COMPARISON")
        print("-" * 30)
        
        # Merge feature importances
        xgb_imp = xgb_results['feature_importance'].head(10).set_index('feature')['importance']
        rf_imp = rf_results['feature_importance'].head(10).set_index('feature')['importance']
        
        importance_comparison = pd.DataFrame({
            'XGBoost': xgb_imp,
            'Random_Forest': rf_imp
        }).fillna(0)
        
        print("Top features comparison:")
        print(importance_comparison.head(10).round(4))
    
    return models_comparison, best_model_name

# Compare all models
models_comparison, best_model_name = compare_all_models(xgb_results, svm_results, rf_results, nn_results, model_data)


📊 MODEL COMPARISON ANALYSIS
🏆 MODEL PERFORMANCE RANKING
------------------------------
1. XGBoost:
   Test Accuracy: 0.6397
   CV Score: 0.6376 (±0.0123)
   Overfitting: 0.2524

2. Random Forest:
   Test Accuracy: 0.6055
   CV Score: 0.6162 (±0.0094)
   Overfitting: 0.2453

3. SVM:
   Test Accuracy: 0.5836
   CV Score: 0.5910 (±0.0033)
   Overfitting: 0.1249

4. Neural Network:
   Test Accuracy: 0.5762
   CV Score: 0.5913 (±0.0061)
   Overfitting: 0.2182

🥇 BEST MODEL: XGBoost
   Accuracy: 0.6397
   Improvement over baseline: 0.1968
   Improvement percentage: 44.4%

🎯 FEATURE IMPORTANCE COMPARISON
------------------------------
Top features comparison:
                             XGBoost  Random_Forest
feature                                            
away_form_10_goals_for_avg    0.0000         0.0150
away_stat_assists             0.0597         0.0707
away_stat_goals               0.0833         0.1193
away_stat_redcards            0.0149         0.0000
form_10_goal_diff_advantage

## 💰 Step 3.7: Betting Simulation

Let's simulate betting performance using our best model to see real-world profitability.

**Betting simulation:**
- Kelly Criterion for bet sizing
- Minimum probability threshold
- Return on Investment (ROI) calculation
- Win rate analysis
"""

In [46]:
# CELL 16 - CODE: Betting Performance Simulation
def simulate_betting_performance(best_model_name, xgb_results, svm_results, rf_results, nn_results, model_data):
    """Simulate betting performance with the best model"""
    
    print("💰 BETTING PERFORMANCE SIMULATION")
    print("=" * 50)
    
    # Get best model results
    model_map = {
        'XGBoost': xgb_results,
        'SVM': svm_results,
        'Random Forest': rf_results,
        'Neural Network': nn_results
    }
    
    best_results = model_map[best_model_name]
    predictions = best_results['predictions']
    probabilities = best_results['probabilities']
    
    # Get actual results
    y_test = model_data['y_test']
    le = model_data['label_encoder']
    
    # Convert back to original labels
    actual_labels = le.inverse_transform(y_test)
    predicted_labels = le.inverse_transform(predictions)
    
    # Simulate betting with confidence threshold
    confidence_threshold = 0.6  # Only bet when model is 60%+ confident
    bankroll = 1000  # Starting bankroll
    bet_size = 0.02  # 2% of bankroll per bet
    
    total_bets = 0
    winning_bets = 0
    total_profit = 0
    bet_history = []
    
    # Typical bookmaker odds (simplified)
    odds_map = {'H': 2.0, 'D': 3.3, 'A': 2.5}  # Home, Draw, Away odds
    
    for i, (actual, predicted, probs) in enumerate(zip(actual_labels, predicted_labels, probabilities)):
        max_prob = np.max(probs)
        
        if max_prob >= confidence_threshold:
            # Place bet
            total_bets += 1
            bet_amount = bankroll * bet_size
            
            # Calculate potential payout
            predicted_class = le.classes_[np.argmax(probs)]
            odds = odds_map[predicted_class]
            
            if actual == predicted_class:
                # Winning bet
                winning_bets += 1
                profit = bet_amount * (odds - 1)
                total_profit += profit
                bet_history.append({
                    'bet': bet_amount,
                    'odds': odds,
                    'profit': profit,
                    'confidence': max_prob
                })
            else:
                # Losing bet
                total_profit -= bet_amount
                bet_history.append({
                    'bet': bet_amount,
                    'odds': odds,
                    'profit': -bet_amount,
                    'confidence': max_prob
                })
    
    # Calculate metrics
    win_rate = winning_bets / total_bets if total_bets > 0 else 0
    roi = (total_profit / (total_bets * bankroll * bet_size)) * 100 if total_bets > 0 else 0
    final_bankroll = bankroll + total_profit
    
    print(f"🎯 BETTING SIMULATION RESULTS")
    print(f"   Model used: {best_model_name}")
    print(f"   Confidence threshold: {confidence_threshold}")
    print(f"   Total test matches: {len(y_test):,}")
    print(f"   Bets placed: {total_bets:,} ({total_bets/len(y_test)*100:.1f}%)")
    print(f"   Winning bets: {winning_bets:,}")
    print(f"   Win rate: {win_rate:.3f}")
    print(f"   Total profit: ${total_profit:.2f}")
    print(f"   ROI: {roi:.2f}%")
    print(f"   Final bankroll: ${final_bankroll:.2f}")
    
    # Profitability analysis
    if roi > 0:
        print(f"   🎉 PROFITABLE STRATEGY!")
    else:
        print(f"   ⚠️  Strategy needs refinement")
    
    return {
        'total_bets': total_bets,
        'winning_bets': winning_bets,
        'win_rate': win_rate,
        'total_profit': total_profit,
        'roi': roi,
        'final_bankroll': final_bankroll,
        'bet_history': bet_history
    }

# Simulate betting performance
betting_results = simulate_betting_performance(best_model_name, xgb_results, svm_results, rf_results, nn_results, model_data)


💰 BETTING PERFORMANCE SIMULATION
🎯 BETTING SIMULATION RESULTS
   Model used: XGBoost
   Confidence threshold: 0.6
   Total test matches: 4,061
   Bets placed: 2,023 (49.8%)
   Winning bets: 1,656
   Win rate: 0.819
   Total profit: $33036.00
   ROI: 81.65%
   Final bankroll: $34036.00
   🎉 PROFITABLE STRATEGY!


## 💾 Step 3.8: Save Trained Models

Let's save our trained models and results for use in the Streamlit dashboard.

**Saved artifacts:**
- Best performing model
- Model comparison results
- Betting simulation results
- Feature importance data
"""


In [48]:
# CELL 18 - CODE: Save Models and Results
def save_trained_models_and_results(xgb_results, svm_results, rf_results, nn_results, 
                                   models_comparison, betting_results, model_data, best_model_name):
    """Save all trained models and results"""
    
    print("💾 SAVING TRAINED MODELS AND RESULTS")
    print("=" * 50)
    
    # Prepare models dictionary
    trained_models = {
        'xgboost': {
            'model': xgb_results['model'],
            'results': xgb_results,
            'type': 'tree'
        },
        'svm': {
            'model': svm_results['model'],
            'results': svm_results,
            'type': 'svm'
        },
        'random_forest': {
            'model': rf_results['model'],
            'results': rf_results,
            'type': 'tree'
        },
        'neural_network': {
            'model': nn_results['model'],
            'results': nn_results,
            'type': 'neural'
        }
    }
    
    # Model metadata
    model_metadata = {
        'best_model': best_model_name.lower().replace(' ', '_'),
        'feature_names': model_data['feature_names'],
        'label_encoder': model_data['label_encoder'],
        'scaler': model_data['scaler'],
        'models_comparison': models_comparison,
        'betting_results': betting_results,
        'baseline_accuracy': model_data['baseline_accuracy'],
        'training_samples': len(model_data['X_train']),
        'test_samples': len(model_data['X_test'])
    }
    
    # Save everything
    with open('trained_models.pkl', 'wb') as f:
        pickle.dump(trained_models, f)
    
    with open('model_metadata.pkl', 'wb') as f:
        pickle.dump(model_metadata, f)
    
    print("✅ Models and results saved successfully!")
    print(f"   trained_models.pkl: All 4 trained models")
    print(f"   model_metadata.pkl: Comparison results and metadata")
    
    # Summary for Streamlit preparation
    streamlit_summary = {
        'best_model': best_model_name,
        'best_accuracy': models_comparison.iloc[0]['Test_Accuracy'],
        'total_features': len(model_data['feature_names']),
        'betting_roi': betting_results['roi'],
        'betting_win_rate': betting_results['win_rate']
    }
    
    with open('streamlit_summary.pkl', 'wb') as f:
        pickle.dump(streamlit_summary, f)
    
    print(f"   streamlit_summary.pkl: Key metrics for dashboard")
    
    return streamlit_summary

# Save all models and results
streamlit_summary = save_trained_models_and_results(
    xgb_results, svm_results, rf_results, nn_results, 
    models_comparison, betting_results, model_data, best_model_name
)


💾 SAVING TRAINED MODELS AND RESULTS
✅ Models and results saved successfully!
   trained_models.pkl: All 4 trained models
   model_metadata.pkl: Comparison results and metadata
   streamlit_summary.pkl: Key metrics for dashboard


## 🎉 Step 3 Complete!

### 🏆 **Training Results Summary:**

**Models Trained:**
- ✅ XGBoost (Gradient Boosting)
- ✅ SVM (Support Vector Machine) 
- ✅ Random Forest (Ensemble)
- ✅ Neural Network (Deep Learning)

**Best Model Performance:**
- 🥇 Model: [Best model will be shown after training]
- 📊 Accuracy: [Accuracy will be shown]
- 💰 Betting ROI: [ROI will be shown]

**Ready for Streamlit Dashboard:**
- All models saved and ready for deployment
- Feature engineering pipeline preserved
- Betting simulation logic implemented
- Model comparison data available

### 🚀 **Next Step: Streamlit Dashboard**

We'll create an interactive web application that:
1. **Live Predictions** - Input team data for match predictions
2. **Model Comparison** - Interactive comparison of all models
3. **Betting Analysis** - ROI simulation and value betting detection
4. **Feature Insights** - Understanding what drives predictions
"""

In [49]:
# CELL 20 - CODE: Step 3 Summary and Next Steps
def display_final_summary(streamlit_summary, models_comparison, betting_results):
    """Display final summary of Step 3 results"""
    
    print("🎉 STEP 3 COMPLETE - MACHINE LEARNING TRAINING FINISHED!")
    print("=" * 60)
    
    print(f"✅ All 4 models trained and evaluated successfully!")
    print(f"✅ Best model identified: {streamlit_summary['best_model']}")
    print(f"✅ Achieved {streamlit_summary['best_accuracy']:.4f} accuracy")
    print(f"✅ Betting simulation: {streamlit_summary['betting_roi']:.2f}% ROI")
    print(f"✅ Models saved for Streamlit deployment")
    
    print(f"\n📊 FINAL PERFORMANCE SUMMARY:")
    print("-" * 40)
    
    for i, (_, row) in enumerate(models_comparison.iterrows(), 1):
        status = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "📊"
        print(f"{status} {row['Model']}: {row['Test_Accuracy']:.4f} accuracy")
    
    print(f"\n💰 BETTING SIMULATION RESULTS:")
    print("-" * 40)
    print(f"   Total bets placed: {betting_results['total_bets']:,}")
    print(f"   Win rate: {betting_results['win_rate']:.3f}")
    print(f"   ROI: {betting_results['roi']:.2f}%")
    print(f"   Profit: ${betting_results['total_profit']:.2f}")
    
    if betting_results['roi'] > 0:
        print(f"   🎉 PROFITABLE BETTING STRATEGY ACHIEVED!")
    else:
        print(f"   📈 Strategy shows promise - refinement needed")
    
    print(f"\n🚀 READY FOR STREAMLIT DASHBOARD!")
    print("=" * 60)
    print("Next: Create interactive web application for live predictions")
    
    return True

# Display final summary
display_final_summary(streamlit_summary, models_comparison, betting_results)

print(f"\n📁 Files created for Streamlit:")
print("   • trained_models.pkl")
print("   • model_metadata.pkl") 
print("   • streamlit_summary.pkl")
print("   • ml_data_prepared.pkl (from Step 2)")
print("   • final_dataset.csv (from Step 2)")


🎉 STEP 3 COMPLETE - MACHINE LEARNING TRAINING FINISHED!
✅ All 4 models trained and evaluated successfully!
✅ Best model identified: XGBoost
✅ Achieved 0.6397 accuracy
✅ Betting simulation: 81.65% ROI
✅ Models saved for Streamlit deployment

📊 FINAL PERFORMANCE SUMMARY:
----------------------------------------
🥇 XGBoost: 0.6397 accuracy
🥈 Random Forest: 0.6055 accuracy
🥉 SVM: 0.5836 accuracy
📊 Neural Network: 0.5762 accuracy

💰 BETTING SIMULATION RESULTS:
----------------------------------------
   Total bets placed: 2,023
   Win rate: 0.819
   ROI: 81.65%
   Profit: $33036.00
   🎉 PROFITABLE BETTING STRATEGY ACHIEVED!

🚀 READY FOR STREAMLIT DASHBOARD!
Next: Create interactive web application for live predictions

📁 Files created for Streamlit:
   • trained_models.pkl
   • model_metadata.pkl
   • streamlit_summary.pkl
   • ml_data_prepared.pkl (from Step 2)
   • final_dataset.csv (from Step 2)
